# TOFU 3B NPO-LoRA profile-cohort recovery diagnostic

This is a post-hoc exploratory follow-up to the previous 3B one-seed run, whose four utility gates failed. It does not change that result. The locked protocol selects the smallest retain-NLL weight in `[5, 10, 20]` that passes every development gate, then evaluates one adapter on a separate final profile cohort. The run stops without final training if no development candidate passes.

The dataset audit verifies pinned source hashes, contiguous profile assignments, and exclusion of the two conflicting-question profiles before model weights load. Metrics are TOFU answer probability, greedy-generation ROUGE-L recall, and forget Truth Ratio. This profile subset is not a standard full-split leaderboard evaluation and a one-seed result is not confirmatory.

Select Kaggle GPU T4 x2 and enable Internet before running all cells. The notebook uses public pinned Open-Unlearning model weights; it needs no Hugging Face token. Full training is enabled because the purpose is to execute the preregistered diagnostic.

In [ ]:
import importlib.metadata as metadata
import json, os, subprocess, sys, urllib.request
from pathlib import Path
print({'notebook': 'profile-cohort recovery', 'protocol': 'locked', 'training_will_start_after_gates': True})


In [ ]:
requirements = Path('/kaggle/working/requirements-npo-pilot.txt')
requirements.write_text('# Kaggle runtime packages used by the bounded TOFU NPO-LoRA pilot.\n# Keep the preinstalled CUDA-enabled torch build supplied by Kaggle.\ntransformers==5.22.1\naccelerate==1.15.0\npeft==0.21.2\nscipy==1.16.3\nrouge-score==0.1.2\n', encoding='utf-8')
expected = {'transformers': '5.22.1', 'accelerate': '1.15.0', 'peft': '0.21.2', 'scipy': '1.16.3', 'rouge-score': '0.1.2'}
def installed_version(package):
    try: return metadata.version(package)
    except metadata.PackageNotFoundError: return None
if any(installed_version(package) != version for package, version in expected.items()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(requirements)], check=True)
versions = {package: metadata.version(package) for package in expected}
if versions != expected: raise RuntimeError(f'Pinned dependency verification failed: {versions}')
try:
    torchao_version = metadata.version('torchao')
except metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '--yes', 'torchao'], check=True, capture_output=True, text=True)
    try: metadata.version('torchao'); raise RuntimeError('torchao remains installed')
    except metadata.PackageNotFoundError: pass
print({'packages': versions, 'torch': metadata.version('torch'), 'torchao': 'absent'})


In [ ]:
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 x2. No model weights are loaded before this check.')
gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
if gpu_names.count('Tesla T4') != 2: raise RuntimeError(f'Expected T4 x2, found {gpu_names}')
with urllib.request.urlopen('https://huggingface.co', timeout=20) as response: internet_status = response.status
if internet_status != 200: raise RuntimeError(f'Hugging Face connectivity failed: HTTP {internet_status}')
print({'gpu_names': gpu_names, 'gpu_memory_bytes': [torch.cuda.get_device_properties(i).total_memory for i in range(2)], 'huggingface_http': internet_status})


In [ ]:
upstream = Path('/kaggle/working/open-unlearning')
commit = '2d98027677f3468d52b6baa3fc4f2b9f2dc08820'
if not upstream.exists(): subprocess.run(['git', 'clone', '--filter=blob:none', 'https://github.com/locuslab/open-unlearning.git', str(upstream)], check=True)
subprocess.run(['git', '-C', str(upstream), 'fetch', '--depth', '1', 'origin', commit], check=True)
subprocess.run(['git', '-C', str(upstream), 'checkout', '--detach', commit], check=True)
actual = subprocess.check_output(['git', '-C', str(upstream), 'rev-parse', 'HEAD'], text=True).strip()
if actual != commit: raise RuntimeError(f'Open-Unlearning commit mismatch: {actual}')
print({'open_unlearning_commit': actual})


In [ ]:
%%writefile /kaggle/working/tofu_official_eval.py
#!/usr/bin/env python3
"""Evaluate pinned TOFU checkpoints with the core Open-Unlearning metrics.

This is inference-only. Full checkpoints receive answer likelihood, greedy
generation/ROUGE-L, and truth-ratio evaluation on the canonical TOFU sets.
The paired retain90 oracle checkpoints are scored on forget truth ratios only
to compute the benchmark's forget-quality comparison.
"""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import os
import random
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import torch
import torch.nn.functional as F
from rouge_score import rouge_scorer
from scipy.stats import ks_2samp
from transformers import AutoModelForCausalLM, AutoTokenizer


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
UPSTREAM_REVISION = "2d98027677f3468d52b6baa3fc4f2b9f2dc08820"
PROFILE_QA_COUNT = 20
BOOTSTRAP_SEED = 42
BOOTSTRAP_DRAWS = 5000
MAX_LENGTH = 512
SYSTEM_PROMPT = "You are a helpful assistant."
TEMPLATE_DATE = "10 Apr 2025"
MAX_NEW_TOKENS = 200
IGNORE_INDEX = -100
EPSILON = 1e-10

CHECKPOINTS = {
    "llama_3_2_3b_full": {
        "label": "Llama 3.2 3B full",
        "family": "llama_3_2_3b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
        "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    },
    "llama_3_2_3b_retain90": {
        "label": "Llama 3.2 3B retain90 oracle",
        "family": "llama_3_2_3b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
        "revision": "2c35819e9a731d096df9abd6efbcc361870c5333",
    },
    "llama_3_1_8b_full": {
        "label": "Llama 3.1 8B full",
        "family": "llama_3_1_8b",
        "role": "full",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_full",
        "revision": "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e",
    },
    "llama_3_1_8b_retain90": {
        "label": "Llama 3.1 8B retain90 oracle",
        "family": "llama_3_1_8b",
        "role": "retain90_oracle",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90",
        "revision": "f02956a9f25a18f38ad7808fbefb2f957347273e",
    },
}

DATA_FILES = (
    "forget10.json",
    "forget10_perturbed.json",
    "retain90.json",
    "retain_perturbed.json",
    "holdout10.json",
    "real_authors_perturbed.json",
    "world_facts_perturbed.json",
)


def get_bytes(url: str) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-official-eval/1.0"})
    with urlopen(request, timeout=60) as response:
        return response.read()


def load_data() -> tuple[dict[str, list[dict]], dict[str, str]]:
    rows_by_file = {}
    hashes = {}
    for filename in DATA_FILES:
        url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
        raw = get_bytes(url)
        rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
        if any(not isinstance(row, dict) for row in rows):
            raise ValueError(f"{filename} contains a non-object JSONL row")
        if any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
            raise ValueError(f"{filename} has a missing question/answer string")
        rows_by_file[filename] = rows
        hashes[filename] = hashlib.sha256(raw).hexdigest()
    return rows_by_file, hashes


def build_eval_sets(data: dict[str, list[dict]]) -> dict[str, list[dict]]:
    """Build canonical evaluation rows and attach source profile blocks."""
    retain_source = data["retain90.json"]
    retain_index = collections.defaultdict(list)
    retain_question_answers = collections.defaultdict(set)
    for i, row in enumerate(retain_source):
        retain_index[(row["question"], row["answer"])].append(i)
        retain_question_answers[row["question"]].add(row["answer"])
    collision_indices = {
        i
        for i, row in enumerate(retain_source)
        if len(retain_question_answers[row["question"]]) > 1
    }

    def basic(rows: list[dict], split: str, source_indices: list[int] | None = None) -> list[dict]:
        result = []
        for i, row in enumerate(rows):
            source_i = i if source_indices is None else source_indices[i]
            result.append({
                "split": split,
                "eval_row_index": i,
                "source_row_index": source_i,
                "profile_block": source_i // PROFILE_QA_COUNT if split in {"forget10", "retain90", "holdout10"} else None,
                "question": row["question"],
                "answer": row["answer"],
                "ambiguous_question_collision": split == "retain90" and source_i in collision_indices,
                "paraphrased_answer": row.get("paraphrased_answer"),
                "perturbed_answers": row.get("perturbed_answer"),
            })
        return result

    forget = data["forget10_perturbed.json"]
    forget_source = data["forget10.json"]
    if len(forget) != len(forget_source) or any(
        (a["question"], a["answer"]) != (b["question"], b["answer"])
        for a, b in zip(forget_source, forget)
    ):
        raise ValueError("forget10_perturbed no longer matches forget10 in source order")

    retain = data["retain_perturbed.json"]
    retain_source_indices = []
    for row in retain:
        matches = retain_index.get((row["question"], row["answer"]), [])
        if len(matches) != 1:
            raise ValueError("A canonical retain_perturbed row does not map uniquely to retain90")
        retain_source_indices.append(matches[0])
    if sorted(retain_source_indices) != list(range(20 * PROFILE_QA_COUNT)):
        raise ValueError("Canonical retain_perturbed is not the pinned first-20-profile retain sample")

    return {
        "forget10": basic(forget, "forget10"),
        "retain90": basic(retain, "retain90", retain_source_indices),
        "holdout10": basic(data["holdout10.json"], "holdout10"),
        "real_authors": basic(data["real_authors_perturbed.json"], "real_authors"),
        "world_facts": basic(data["world_facts_perturbed.json"], "world_facts"),
    }


def ids(value) -> list[int]:
    if hasattr(value, "keys") and "input_ids" in value:
        value = value["input_ids"]
    if isinstance(value, torch.Tensor):
        value = value.tolist()
    if value and isinstance(value[0], list):
        value = value[0]
    return [int(x) for x in value]


def encode_answer(tokenizer, question: str, answer: str) -> tuple[list[int], list[int]]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    kwargs = {"date_string": TEMPLATE_DATE}
    full_ids = ids(tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=False, **kwargs
    ))
    prompt_ids = ids(tokenizer.apply_chat_template(
        conversation[:-1], tokenize=True, add_generation_prompt=True, **kwargs
    ))
    if not full_ids or len(prompt_ids) > len(full_ids):
        raise ValueError("Invalid chat-template tokenization")
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokenization is not an exact prefix of the full conversation")
    if tokenizer.eos_token_id is not None and full_ids[-1] != tokenizer.eos_token_id:
        full_ids.append(int(tokenizer.eos_token_id))
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids):]
    if len(labels) != len(full_ids) or not any(x != IGNORE_INDEX for x in labels[1:]):
        raise ValueError("Assistant answer mask is empty or misaligned")
    return full_ids, labels


def encode_prompt(tokenizer, question: str) -> list[int]:
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    result = ids(tokenizer.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        date_string=TEMPLATE_DATE,
    ))
    if not result:
        raise ValueError("Chat-template prompt was empty")
    return result


def collate_scoring(rows: list[tuple[list[int], list[int]]], pad_id: int):
    width = max(len(x[0]) for x in rows)
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    labels = torch.full((len(rows), width), IGNORE_INDEX, dtype=torch.long)
    for i, (tokens, targets) in enumerate(rows):
        n = len(tokens)
        input_ids[i, :n] = torch.tensor(tokens)
        attention[i, :n] = 1
        labels[i, :n] = torch.tensor(targets)
    return input_ids, attention, labels


def score_answers(model, encoded: list[tuple[list[int], list[int]]], pad_id: int, batch_size: int) -> list[dict]:
    device = model.get_input_embeddings().weight.device
    output_scores = []
    offset = 0
    batches = 0
    while offset < len(encoded):
        size = min(batch_size, len(encoded) - offset)
        batch = encoded[offset:offset + size]
        input_ids, attention, labels = collate_scoring(batch, pad_id)
        try:
            with torch.inference_mode():
                out = model(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    use_cache=False,
                )
                logits = out.logits[:, :-1, :].float()
                targets = labels[:, 1:].to(logits.device)
                losses = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="none",
                ).reshape(targets.shape)
                tokens_per_row = (targets != IGNORE_INDEX).sum(dim=1)
                if torch.any(tokens_per_row == 0):
                    raise ValueError("Evaluation example has no scored assistant tokens")
                mean_loss = losses.sum(dim=1) / tokens_per_row
                for i in range(size):
                    value = float(mean_loss[i].item())
                    output_scores.append({
                        "avg_answer_token_loss": value,
                        "answer_probability": math.exp(-value),
                        "scored_answer_tokens": int(tokens_per_row[i].item()),
                    })
            offset += size
            batches += 1
            if batches % 100 == 0 or offset == len(encoded):
                print(f"Scored {offset}/{len(encoded)} answer candidates", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during scoring; reducing batch size to {batch_size}", flush=True)
    return output_scores


def collate_prompts(rows: list[list[int]], pad_id: int):
    width = max(map(len, rows))
    input_ids = torch.full((len(rows), width), pad_id, dtype=torch.long)
    attention = torch.zeros((len(rows), width), dtype=torch.long)
    for i, row in enumerate(rows):
        input_ids[i, -len(row):] = torch.tensor(row)
        attention[i, -len(row):] = 1
    return input_ids, attention


def generate_answers(model, tokenizer, prompts: list[list[int]], batch_size: int) -> list[str]:
    device = model.get_input_embeddings().weight.device
    generations = []
    offset = 0
    batches = 0
    model.config.use_cache = True
    while offset < len(prompts):
        size = min(batch_size, len(prompts) - offset)
        current = prompts[offset:offset + size]
        input_ids, attention = collate_prompts(current, int(tokenizer.pad_token_id))
        try:
            with torch.inference_mode():
                output = model.generate(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    do_sample=False,
                    max_new_tokens=MAX_NEW_TOKENS,
                    use_cache=True,
                    pad_token_id=tokenizer.eos_token_id,
                )
            new_tokens = output[:, input_ids.shape[1]:]
            texts = tokenizer.batch_decode(
                new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            generations.extend(text.strip() for text in texts)
            offset += size
            batches += 1
            if batches % 25 == 0 or offset == len(prompts):
                print(f"Generated {offset}/{len(prompts)} answers", flush=True)
        except torch.cuda.OutOfMemoryError:
            if size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, size // 2)
            print(f"CUDA OOM during generation; reducing batch size to {batch_size}", flush=True)
    return generations


def build_jobs(eval_sets: dict[str, list[dict]], role: str):
    score_jobs = []
    generation_jobs = []
    for split, rows in eval_sets.items():
        for row in rows:
            key = (split, row["eval_row_index"])
            if role == "full" or split == "forget10":
                score_jobs.append({**row, "variant": "answer", "candidate_answer": row["answer"]})
                if row["paraphrased_answer"]:
                    score_jobs.append({**row, "variant": "paraphrased_answer", "candidate_answer": row["paraphrased_answer"]})
                for i, candidate in enumerate(row["perturbed_answers"] or []):
                    score_jobs.append({**row, "variant": f"perturbed_{i}", "candidate_answer": candidate})
            if role == "full":
                generation_jobs.append(row)
    return score_jobs, generation_jobs


def score_rows_by_key(jobs: list[dict], scores: list[dict]) -> dict[tuple[str, int], dict[str, dict]]:
    result = collections.defaultdict(dict)
    for job, score in zip(jobs, scores):
        key = (job["split"], job["eval_row_index"])
        if job["variant"] in result[key]:
            raise ValueError(f"Duplicate answer candidate score for {key}/{job['variant']}")
        result[key][job["variant"]] = score
    return result


def bootstrap_cluster_ci(rows: list[dict], field: str, seed: int = BOOTSTRAP_SEED) -> list[float] | None:
    groups = collections.defaultdict(list)
    for row in rows:
        if row.get("profile_block") is not None and row.get(field) is not None:
            groups[int(row["profile_block"])].append(float(row[field]))
    if not groups:
        return None
    group_values = list(groups.values())
    rng = random.Random(seed)
    estimates = []
    for _ in range(BOOTSTRAP_DRAWS):
        sampled = [rng.choice(group_values) for _ in group_values]
        values = [value for group in sampled for value in group]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return [estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))], estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))]]


def mean_field(rows: list[dict], field: str) -> float | None:
    values = [float(row[field]) for row in rows if row.get(field) is not None]
    return sum(values) / len(values) if values else None


def summarize_examples(examples: list[dict]) -> dict:
    summary = {"n_questions": len(examples), "n_profile_blocks": len({r["profile_block"] for r in examples if r.get("profile_block") is not None})}
    fields = (
        "answer_probability",
        "rougeL_recall",
        "truth_ratio",
        "truth_ratio_score",
        "normalized_option_probability",
    )
    for field in fields:
        value = mean_field(examples, field)
        if value is not None:
            summary[f"mean_{field}"] = value
            ci = bootstrap_cluster_ci(examples, field)
            if ci is not None:
                summary[f"profile_cluster_bootstrap_95pct_ci_{field}"] = ci
    return summary


def make_metrics(
    rows: list[dict],
    split: str,
    score_map: dict[tuple[str, int], dict[str, dict]],
    generations: dict[tuple[str, int], str],
    tokenizer,
    rouge: rouge_scorer.RougeScorer,
) -> list[dict]:
    metrics = []
    for row in rows:
        key = (split, row["eval_row_index"])
        scores = score_map.get(key, {})
        if "answer" not in scores:
            raise ValueError(f"Missing gold-answer score for {key}")
        output = {
            "split": split,
            "eval_row_index": row["eval_row_index"],
            "source_row_index": row["source_row_index"],
            "profile_block": row["profile_block"],
            "question_sha256": hashlib.sha256(row["question"].encode("utf-8")).hexdigest(),
            "source_answer": row["answer"],
            "reference_answer": row["answer"],
            "ambiguous_question_collision": row["ambiguous_question_collision"],
            "answer_probability": scores["answer"]["answer_probability"],
            "answer_token_loss": scores["answer"]["avg_answer_token_loss"],
            "answer_scored_tokens": scores["answer"]["scored_answer_tokens"],
        }
        if "paraphrased_answer" in scores:
            para = scores["paraphrased_answer"]
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"] or []))]
            output["paraphrased_answer_probability"] = para["answer_probability"]
            output["paraphrased_answer_token_loss"] = para["avg_answer_token_loss"]
            output["perturbed_answer_probabilities"] = [x["answer_probability"] for x in wrong]
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            if wrong:
                # Match the pinned Open-Unlearning implementation: exponentiate
                # the mean per-candidate answer loss, i.e. the geometric mean of
                # perturbed-answer probabilities.
                geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
                ratio = geometric_wrong_probability / (para["answer_probability"] + EPSILON)
                output["truth_ratio"] = ratio
                output["truth_ratio_score"] = min(ratio, 1.0 / (ratio + EPSILON)) if split == "forget10" else max(0.0, 1.0 - ratio)
        elif row["perturbed_answers"]:
            wrong = [scores[f"perturbed_{i}"] for i in range(len(row["perturbed_answers"]))]
            wrong_probabilities = [x["answer_probability"] for x in wrong]
            geometric_wrong_probability = math.exp(-sum(x["avg_answer_token_loss"] for x in wrong) / len(wrong))
            ratio = geometric_wrong_probability / (scores["answer"]["answer_probability"] + EPSILON)
            output["perturbed_answer_probabilities"] = wrong_probabilities
            output["perturbed_answer_token_losses"] = [x["avg_answer_token_loss"] for x in wrong]
            output["truth_ratio"] = ratio
            output["truth_ratio_score"] = max(0.0, 1.0 - ratio)
            output["normalized_option_probability"] = scores["answer"]["answer_probability"] / (
                scores["answer"]["answer_probability"] + sum(wrong_probabilities) + EPSILON
            )
        if key in generations:
            text = generations[key]
            full_ids, _ = encode_answer(tokenizer, row["question"], row["answer"])
            prompt_ids = encode_prompt(tokenizer, row["question"])
            input_text = tokenizer.decode(
                prompt_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            full_text = tokenizer.decode(
                full_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            reference = full_text.replace(input_text, "").strip()
            output["generation"] = text
            output["reference_answer"] = reference
            output["rougeL_recall"] = rouge.score(reference, text)["rougeL"].recall
        metrics.append(output)
    return metrics


def model_utility(metrics_by_split: dict[str, list[dict]]) -> dict:
    components = {
        "retain_probability": mean_field(metrics_by_split["retain90"], "answer_probability"),
        "retain_rougeL_recall": mean_field(metrics_by_split["retain90"], "rougeL_recall"),
        "retain_truth_ratio_score": mean_field(metrics_by_split["retain90"], "truth_ratio_score"),
        "real_authors_normalized_option_probability": mean_field(metrics_by_split["real_authors"], "normalized_option_probability"),
        "real_authors_rougeL_recall": mean_field(metrics_by_split["real_authors"], "rougeL_recall"),
        "real_authors_truth_ratio_score": mean_field(metrics_by_split["real_authors"], "truth_ratio_score"),
        "world_facts_normalized_option_probability": mean_field(metrics_by_split["world_facts"], "normalized_option_probability"),
        "world_facts_rougeL_recall": mean_field(metrics_by_split["world_facts"], "rougeL_recall"),
        "world_facts_truth_ratio_score": mean_field(metrics_by_split["world_facts"], "truth_ratio_score"),
    }
    if any(value is None for value in components.values()):
        raise ValueError(f"Cannot compute complete TOFU model utility; missing component: {components}")
    if any(value <= 0 for value in components.values()):
        aggregate = 0.0
    else:
        aggregate = len(components) / sum(1.0 / value for value in components.values())
    return {"components": components, "harmonic_mean": aggregate}


def write_reports(output_dir: Path) -> None:
    keys = ("llama_3_2_3b_full", "llama_3_1_8b_full")
    results = {key: json.loads((output_dir / f"{key}_official_eval.json").read_text(encoding="utf-8")) for key in keys}
    summaries = {key: results[key]["summary"] for key in keys}
    split_labels = {
        "forget10": "Forget10",
        "retain90": "Canonical retain eval",
        "holdout10": "Holdout10",
        "real_authors": "Real authors",
        "world_facts": "World facts",
    }
    lines = [
        "# TOFU official-metric pre-unlearning baseline",
        "",
        "**Status:** complete and inference-only. It evaluates the published full checkpoints and paired retain90 oracle checkpoints. No NPO training or research adapter was run.",
        "",
        "The metric formulas, chat preprocessing, and generated-reference extraction follow the pinned Open-Unlearning implementation. `answer_probability` is mean per-answer-token probability; `rougeL_recall` scores greedy generated answers; Truth Ratio compares the geometric mean likelihood of perturbed answers with the paraphrased correct answer (or source correct answer for real-author/world-fact utility). Kaggle T4 uses FP16 + SDPA here, while upstream configs specify BF16 + FlashAttention 2; formulas and prompts match, but numerical generations are not bitwise-identical to the upstream runtime.",
        "",
        "## Full-checkpoint metrics",
        "",
        "| Split | Model | Questions / profile blocks | Mean answer probability (95% profile CI) | Mean ROUGE-L recall (95% profile CI) | Mean truth ratio | TOFU truth-ratio score | Normalized option probability |",
        "|---|---|---:|---:|---:|---:|---:|---:|",
    ]
    for split, label in split_labels.items():
        for key in keys:
            summary = summaries[key][split]
            def render(metric: str, ci_key: str | None = None) -> str:
                val = summary.get(f"mean_{metric}")
                if val is None:
                    return "—"
                if ci_key and ci_key in summary:
                    low, high = summary[ci_key]
                    return f"{val:.4f} [{low:.4f}, {high:.4f}]"
                return f"{val:.4f}"
            n = f"{summary['n_questions']} / {summary['n_profile_blocks']}" if summary["n_profile_blocks"] else f"{summary['n_questions']} / —"
            lines.append(
                f"| {label} | {results[key]['model']['label']} | {n} | "
                f"{render('answer_probability', 'profile_cluster_bootstrap_95pct_ci_answer_probability')} | "
                f"{render('rougeL_recall', 'profile_cluster_bootstrap_95pct_ci_rougeL_recall')} | "
                f"{render('truth_ratio')} | {render('truth_ratio_score')} | {render('normalized_option_probability')} |"
            )
    lines += ["", "## Composite utility and forget-quality comparison", ""]
    lines.append("| Model | TOFU model-utility harmonic mean | Forget-quality KS p-value vs retain90 oracle |")
    lines.append("|---|---:|---:|")
    for key in keys:
        utility = results[key]["model_utility"]["harmonic_mean"]
        fq = results[key].get("forget_quality_vs_retain90_oracle", {}).get("p_value")
        lines.append(f"| {results[key]['model']['label']} | {utility:.6f} | {fq:.6g}" if fq is not None else f"| {results[key]['model']['label']} | {utility:.6f} | pending oracle result |")

    lines += [
        "",
        "## Retain-question collision sensitivity",
        "",
        "The pinned `retain90` data repeats one generic question with different answers. The canonical 400-row retain evaluation includes one of the two conflicting rows. Primary TOFU metrics retain the official row; the sensitivity row below excludes it.",
        "",
        "| Model | Retain probability, primary → excluding collision | Retain ROUGE-L recall, primary → excluding collision | Retain truth-ratio score, primary → excluding collision |",
        "|---|---:|---:|---:|",
    ]
    for key in keys:
        rows = results[key]["per_example"]["retain90"]
        clean = [row for row in rows if not row["ambiguous_question_collision"]]
        all_summary = summarize_examples(rows)
        clean_summary = summarize_examples(clean)
        lines.append(
            f"| {results[key]['model']['label']} | {all_summary['mean_answer_probability']:.4f} → {clean_summary['mean_answer_probability']:.4f} | "
            f"{all_summary['mean_rougeL_recall']:.4f} → {clean_summary['mean_rougeL_recall']:.4f} | "
            f"{all_summary['mean_truth_ratio_score']:.4f} → {clean_summary['mean_truth_ratio_score']:.4f} |"
        )
    lines += [
        "",
        "## Interpretation limits",
        "",
        "- Confidence intervals resample the 20 inferred profile blocks for forget10, canonical retain, and holdout10. The JSON contains question-level outputs; real-author and world-fact rows have no explicit cluster IDs, so they are reported without profile-cluster intervals.",
        "- Inference uses FP16 + SDPA because Kaggle T4 does not provide the upstream BF16 + FlashAttention 2 runtime. Small numeric and generation differences from an upstream evaluation are possible.",
        "- The 3B and 8B checkpoints come from different Llama generations and training histories. Their paired differences are descriptive checkpoint comparisons, not causal evidence about parameter count.",
        "- A small/large ROUGE or likelihood value alone does not prove forgetting. The forget-quality p-value compares distributions with a retain-only oracle and is not a standalone proof of equivalence or privacy.",
        "- TOFU profiles are fictional. These results cannot establish deletion of real personal data or real-world privacy protection.",
        "- The earlier 6 Nov likelihood-only run used a custom retain sample; this report uses the canonical `retain_perturbed` evaluation set from the pinned benchmark revision.",
        "",
        "## Reproduction metadata",
        "",
        f"Dataset revision: `{DATASET_REVISION}`. Open-Unlearning code revision: `{UPSTREAM_REVISION}`. Bootstrap: {BOOTSTRAP_DRAWS:,} profile-cluster draws, seed {BOOTSTRAP_SEED}. Greedy generation: max {MAX_NEW_TOKENS} new tokens.",
        "",
    ]
    (output_dir / "official_metric_baseline_report.md").write_text("\n".join(lines), encoding="utf-8")

    compare = {}
    for split in split_labels:
        left = results["llama_3_2_3b_full"]["per_example"][split]
        right = results["llama_3_1_8b_full"]["per_example"][split]
        right_by_index = {row["source_row_index"]: row for row in right}
        if len(right_by_index) != len(right) or {row["source_row_index"] for row in left} != set(right_by_index):
            raise ValueError(f"Full checkpoints do not have matched source examples in {split}")
        item = {}
        for metric in ("answer_probability", "rougeL_recall", "truth_ratio", "truth_ratio_score", "normalized_option_probability"):
            pairs = [(row, right_by_index[row["source_row_index"]]) for row in left if metric in row and metric in right_by_index[row["source_row_index"]]]
            if not pairs:
                continue
            deltas = [float(b[metric]) - float(a[metric]) for a, b in pairs]
            metric_result = {"n_questions": len(pairs), "mean_8b_minus_3b": sum(deltas) / len(deltas)}
            grouped = collections.defaultdict(list)
            for (a, _), delta in zip(pairs, deltas):
                if a.get("profile_block") is not None:
                    grouped[int(a["profile_block"])].append(delta)
            if grouped:
                group_ids = list(grouped)
                rng = random.Random(BOOTSTRAP_SEED)
                estimates = []
                for _ in range(BOOTSTRAP_DRAWS):
                    sampled = [rng.choice(group_ids) for _ in group_ids]
                    vals = [value for group in sampled for value in grouped[group]]
                    estimates.append(sum(vals) / len(vals))
                estimates.sort()
                metric_result["profile_cluster_bootstrap_95pct_ci"] = [
                    estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))],
                    estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))],
                ]
            else:
                metric_result["interval"] = None
                metric_result["interval_note"] = "No explicit profile cluster IDs; no inferential interval reported."
            item[metric] = metric_result
        compare[split] = item
    compare["interpretation_limit"] = "Descriptive paired checkpoint contrasts only; model generations/training histories differ."
    (output_dir / "paired_full_checkpoint_comparison.json").write_text(json.dumps(compare, indent=2) + "\n", encoding="utf-8")


def encode_jobs(tokenizer, jobs: list[dict]) -> list[tuple[list[int], list[int]]]:
    return [encode_answer(tokenizer, job["question"], job["candidate_answer"]) for job in jobs]


def evaluate_checkpoint(
    key: str,
    output_dir: Path,
    eval_sets: dict[str, list[dict]],
    dataset_hashes: dict[str, str],
    score_batch_size: int,
    generation_batch_size: int,
    max_memory_per_gpu_gib: float,
    adapter_path: Path | None = None,
    result_key: str | None = None,
    adapter_seed: int | None = None,
    profile_cohort_mode: bool = False,
) -> dict:
    from accelerate import __version__ as accelerate_version
    from transformers import __version__ as transformers_version

    spec = CHECKPOINTS[key]
    cache_root = Path(
        os.environ.get(
            "TOFU_MODEL_CACHE_ROOT",
            str(Path(tempfile.gettempdir()) / "tofu_official_eval_cache"),
        )
    )
    model_cache = cache_root / key
    model_cache.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], cache_dir=str(model_cache), use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if not tokenizer.is_fast:
        raise RuntimeError("Fast tokenizer required")

    score_jobs, generation_jobs = build_jobs(eval_sets, spec["role"])
    encoded_jobs = encode_jobs(tokenizer, score_jobs)
    prompt_ids = [encode_prompt(tokenizer, row["question"]) for row in generation_jobs]
    # Transformers 5.22's allocator warmup reserves a second buffer roughly
    # the size of the 8B model shard. On T4x2 this OOMs before weights load.
    # Disable only this load-time optimization for 8B; checkpoint weights still
    # load normally. Keep the library default for the smaller 3B model.
    allocator_warmup_strategy = "library default"
    if spec["family"] == "llama_3_1_8b":
        import transformers.modeling_utils as modeling_utils

        if not hasattr(modeling_utils, "caching_allocator_warmup"):
            raise RuntimeError("Pinned Transformers version lacks the expected allocator warmup hook")
        modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None
        allocator_warmup_strategy = "disabled for 8B to avoid duplicate preallocation; weights load normally"

    memory_limit = f"{max_memory_per_gpu_gib:g}GiB"
    # With only two T4s, balanced_low_0 put almost all 8B decoder blocks on
    # GPU 1 and caused CPU offload. Even balancing leaves ample generation
    # headroom on both GPUs at the configured 13 GiB cap.
    print(f"Loading {spec['label']} with balanced; max_memory={memory_limit} per T4", flush=True)
    model = AutoModelForCausalLM.from_pretrained(
        spec["repo_id"],
        revision=spec["revision"],
        cache_dir=str(model_cache),
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="balanced",
        max_memory={0: memory_limit, 1: memory_limit, "cpu": "40GiB"},
        attn_implementation="sdpa",
    )
    device_map = getattr(model, "hf_device_map", {})
    offloaded_devices = sorted({str(device) for device in device_map.values() if str(device).lower() in {"cpu", "disk"}})
    print({"loaded_model": spec["label"], "hf_device_map": device_map, "offloaded_devices": offloaded_devices}, flush=True)
    if spec["family"] == "llama_3_1_8b" and offloaded_devices:
        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise RuntimeError(f"8B placement used CPU/disk offload {offloaded_devices}; refusing a slow, unbounded evaluation")
    result_spec = dict(spec)
    if adapter_path is not None:
        from peft import PeftModel

        if not (adapter_path / "adapter_config.json").is_file():
            raise FileNotFoundError(f"LoRA adapter config missing from {adapter_path}")
        model = PeftModel.from_pretrained(model, str(adapter_path), is_trainable=False)
        result_spec["label"] = f"{spec['label']} NPO LoRA seed {adapter_seed}"
        result_spec["adapter"] = {
            "method": "NPO + retain NLL, LoRA adapter",
            "seed": adapter_seed,
            "adapter_dir_name": adapter_path.name,
            "files_sha256": {
                p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                for p in sorted(adapter_path.iterdir()) if p.is_file()
            },
        }
    model.eval()
    model.config.use_cache = False

    # A tiny non-research prompt tests generation before the full dataset pass.
    smoke_ids = encode_prompt(tokenizer, "Reply with one short greeting.")
    smoke_tensor = torch.tensor([smoke_ids], dtype=torch.long, device=model.get_input_embeddings().weight.device)
    smoke_started = time.monotonic()
    model.config.use_cache = True
    with torch.inference_mode():
        smoke_output = model.generate(
            input_ids=smoke_tensor,
            attention_mask=torch.ones_like(smoke_tensor),
            do_sample=False,
            max_new_tokens=4,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    model.config.use_cache = False
    smoke_new_tokens = int(smoke_output.shape[1] - smoke_tensor.shape[1])
    if smoke_new_tokens < 1:
        raise RuntimeError("GPU placement smoke generated no tokens; refusing full evaluation")
    placement_smoke = {
        "status": "PASS",
        "new_tokens": smoke_new_tokens,
        "elapsed_seconds": round(time.monotonic() - smoke_started, 2),
        "research_rows_used": 0,
    }
    print({"placement_smoke": placement_smoke}, flush=True)

    print(f"Starting answer scoring: {len(encoded_jobs)} candidates", flush=True)
    score_values = score_answers(model, encoded_jobs, int(tokenizer.pad_token_id), score_batch_size)
    if len(score_values) != len(score_jobs):
        raise RuntimeError("Scored row count does not equal job count")
    score_map = score_rows_by_key(score_jobs, score_values)
    generations = {}
    if spec["role"] == "full":
        print(f"Starting greedy generation: {len(prompt_ids)} prompts", flush=True)
        generated_texts = generate_answers(model, tokenizer, prompt_ids, generation_batch_size)
        if len(generated_texts) != len(generation_jobs):
            raise RuntimeError("Generation row count does not equal prompt count")
        generations = {
            (row["split"], row["eval_row_index"]): text
            for row, text in zip(generation_jobs, generated_texts)
        }

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge.tokenizer = tokenizer
    splits_to_write = ("forget10",) if spec["role"] == "retain90_oracle" else tuple(eval_sets)
    metrics_by_split = {
        split: make_metrics(eval_sets[split], split, score_map, generations, tokenizer, rouge)
        for split in splits_to_write
    }
    result = {
        "status": (
            "PASS_PROFILE_COHORT_EVAL" if profile_cohort_mode
            else "PASS_OFFICIAL_TOFU_ADAPTER_EVAL" if adapter_path is not None
            else "PASS_OFFICIAL_TOFU_BASELINE"
        ),
        "model": result_spec,
        **({"study_type": "exploratory profile-cohort follow-up; not a standard full-split TOFU evaluation"} if profile_cohort_mode else {}),
        "evaluation": {
            "dataset_id": DATASET_ID,
            "dataset_revision": DATASET_REVISION,
            "dataset_sha256_by_file": dataset_hashes,
            "open_unlearning_revision": UPSTREAM_REVISION,
            "profile_block_size": PROFILE_QA_COUNT,
            "bootstrap_seed": BOOTSTRAP_SEED,
            "bootstrap_draws": BOOTSTRAP_DRAWS,
            "generation": {"do_sample": False, "max_new_tokens": MAX_NEW_TOKENS, "use_cache": True},
            "declared_max_input_length": MAX_LENGTH,
            "input_truncation": False,
            "max_scored_sequence_tokens": max((len(tokens) for tokens, _ in encoded_jobs), default=0),
            "scored_sequences_over_declared_max_length": sum(len(tokens) > MAX_LENGTH for tokens, _ in encoded_jobs),
            "max_generation_prompt_tokens": max((len(tokens) for tokens in prompt_ids), default=0),
            **({
                "profile_cohort_mode": True,
                "rows_by_split": {split: len(rows) for split, rows in eval_sets.items()},
                "profile_blocks_by_split": {
                    split: sorted({row["profile_block"] for row in rows if row.get("profile_block") is not None})
                    for split, rows in eval_sets.items()
                },
                "metric_implementation": "Pinned TOFU answer probability, greedy-generation ROUGE-L recall, and forget Truth Ratio; profile-cohort subset, not a standard full-split benchmark evaluation.",
            } if profile_cohort_mode else {
                "canonical_retain_eval_rows": len(eval_sets["retain90"]),
                "canonical_retain_profile_blocks": sorted({row["profile_block"] for row in eval_sets["retain90"]}),
                "metric_implementation": "Pinned Open-Unlearning probability, ROUGE-L recall, Truth Ratio, normalized option probability, model-utility harmonic mean, and KS forget-quality definitions; see README for exact conventions.",
            }),
            "score_jobs": len(score_jobs),
            "generation_jobs": len(generation_jobs),
        },
        "summary": {split: summarize_examples(rows) for split, rows in metrics_by_split.items()},
        "per_example": {split: rows for split, rows in metrics_by_split.items()},
        "model_utility": model_utility(metrics_by_split) if spec["role"] == "full" and not profile_cohort_mode else None,
        "runtime": {
            "torch_version": torch.__version__,
            "transformers_version": transformers_version,
            "accelerate_version": accelerate_version,
            "rouge_score_version": __import__("importlib.metadata", fromlist=["version"]).version("rouge-score"),
            "scipy_version": __import__("scipy").__version__,
            "inference_dtype": "float16",
            "max_memory_per_gpu_gib": max_memory_per_gpu_gib,
            "device_map_strategy": "balanced",
            "hf_device_map": {str(k): str(v) for k, v in device_map.items()},
            "cpu_offload_used": bool(offloaded_devices),
            "allocator_warmup_strategy": allocator_warmup_strategy,
            "placement_smoke": placement_smoke,
            "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "gpu_total_memory_bytes": [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())],
            "elapsed_seconds": round(time.monotonic() - started, 2),
            "training_started": False,
            "research_adapter_created": adapter_path is not None,
        },
    }
    result_key = result_key or key
    suffix = "_profile_eval.json" if profile_cohort_mode else "_official_eval.json"
    result_path = output_dir / f"{result_key}{suffix}"
    result_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    score_path = output_dir / f"{result_key}_examples.jsonl"
    with score_path.open("w", encoding="utf-8") as f:
        for rows in metrics_by_split.values():
            for row in rows:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    del model, tokenizer, encoded_jobs, prompt_ids
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(model_cache, ignore_errors=True)
    return result


def add_forget_quality(results_dir: Path, full_key: str, oracle_key: str) -> None:
    full_path = results_dir / f"{full_key}_official_eval.json"
    oracle_path = results_dir / f"{oracle_key}_official_eval.json"
    if not full_path.exists() or not oracle_path.exists():
        return
    full = json.loads(full_path.read_text(encoding="utf-8"))
    oracle = json.loads(oracle_path.read_text(encoding="utf-8"))
    full_rows = full["per_example"]["forget10"]
    oracle_rows = oracle["per_example"]["forget10"]
    if [(r["source_row_index"], r["question_sha256"]) for r in full_rows] != [
        (r["source_row_index"], r["question_sha256"]) for r in oracle_rows
    ]:
        raise ValueError("Full and retain90 oracle forget rows are not aligned")
    stat = ks_2samp(
        [1.0 / (row["truth_ratio"] + EPSILON) for row in full_rows],
        [1.0 / (row["truth_ratio"] + EPSILON) for row in oracle_rows],
    )
    full["forget_quality_vs_retain90_oracle"] = {
        "metric": "KS p-value on inverse forget truth-ratio distributions; monotonic inversion leaves the two-sample KS statistic and p-value unchanged",
        "n_full_checkpoint": len(full_rows),
        "n_retain90_oracle": len(oracle_rows),
        "ks_statistic": float(stat.statistic),
        "p_value": float(stat.pvalue),
        "interpretation": "A larger p-value means these two empirical distributions are less distinguishable by this KS test; it is not proof of successful unlearning.",
    }
    full_path.write_text(json.dumps(full, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/tofu_official_eval"))
    parser.add_argument("--models", default=",".join(CHECKPOINTS))
    parser.add_argument("--score-batch-size", type=int, default=2)
    parser.add_argument("--generation-batch-size", type=int, default=2)
    parser.add_argument("--max-memory-per-gpu-gib", type=float, default=13.0)
    parser.add_argument("--resume-existing", action="store_true", help="Reuse passing, pinned result files already in output-dir; final verifier checks every row and metric.")
    args = parser.parse_args()
    selected = [x.strip() for x in args.models.split(",") if x.strip()]
    if not selected or set(selected) - set(CHECKPOINTS):
        parser.error(f"models must be selected from {sorted(CHECKPOINTS)}")
    if args.max_memory_per_gpu_gib <= 0 or args.max_memory_per_gpu_gib >= 14:
        parser.error("max-memory-per-gpu-gib must be >0 and <14 to leave T4 runtime headroom")
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("Requires GPU T4 x2; stopped before loading model weights")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    data, hashes = load_data()
    eval_sets = build_eval_sets(data)
    status_path = args.output_dir / "official_eval_status.json"
    reused_results = {}
    if args.resume_existing and status_path.exists():
        previous = json.loads(status_path.read_text(encoding="utf-8"))
        if previous.get("dataset_hashes") != hashes:
            raise RuntimeError("Cannot resume: existing receipt has different pinned dataset hashes")
        for key, entry in previous.get("model_results", {}).items():
            if entry.get("status") != "PASS_OFFICIAL_TOFU_BASELINE":
                continue
            result_path = args.output_dir / f"{key}_official_eval.json"
            examples_path = args.output_dir / f"{key}_examples.jsonl"
            if key not in CHECKPOINTS or not result_path.is_file() or not examples_path.is_file():
                raise RuntimeError(f"Cannot resume: incomplete successful receipt for {key}")
            prior_result = json.loads(result_path.read_text(encoding="utf-8"))
            spec = CHECKPOINTS[key]
            if (
                prior_result.get("status") != "PASS_OFFICIAL_TOFU_BASELINE"
                or prior_result.get("model", {}).get("repo_id") != spec["repo_id"]
                or prior_result.get("model", {}).get("revision") != spec["revision"]
                or prior_result.get("model", {}).get("role") != spec["role"]
                or prior_result.get("evaluation", {}).get("dataset_sha256_by_file") != hashes
            ):
                raise RuntimeError(f"Cannot resume: pinned model/data metadata mismatch for {key}")
            reused_results[key] = entry
        if set(selected) & set(reused_results):
            raise RuntimeError("Requested model is already marked successful; remove it from --models or start a clean output directory")

    for key in selected:
        if CHECKPOINTS[key]["role"] == "full":
            oracle_key = key.replace("_full", "_retain90")
            if oracle_key not in selected and oracle_key not in reused_results:
                parser.error(f"{key} requires its matching retain90 oracle in --models or a verified resumed output")

    status = {
        "status": "RUNNING",
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "selected_models": selected,
        "reused_models": sorted(reused_results),
        "dataset_hashes": hashes,
        "model_results": reused_results,
        "training_started": False,
    }
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    for key in selected:
        try:
            result = evaluate_checkpoint(
                key,
                args.output_dir,
                eval_sets,
                hashes,
                args.score_batch_size,
                args.generation_batch_size,
                args.max_memory_per_gpu_gib,
            )
            status["model_results"][key] = {
                "status": result["status"],
                "result_file": f"{key}_official_eval.json",
                "examples_file": f"{key}_examples.jsonl",
                "runtime_seconds": result["runtime"]["elapsed_seconds"],
            }
        except Exception as error:
            shutil.rmtree(Path(tempfile.gettempdir()) / "tofu_official_eval_cache" / key, ignore_errors=True)
            status["status"] = "ERROR"
            status["error"] = f"{type(error).__name__}: {error}"
            status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            raise
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"{key}: PASS_OFFICIAL_TOFU_BASELINE", flush=True)
    for full_key, oracle_key in (
        ("llama_3_2_3b_full", "llama_3_2_3b_retain90"),
        ("llama_3_1_8b_full", "llama_3_1_8b_retain90"),
    ):
        if full_key in status["model_results"] and oracle_key in status["model_results"]:
            add_forget_quality(args.output_dir, full_key, oracle_key)
    complete_keys = set(CHECKPOINTS)
    if complete_keys <= set(status["model_results"]):
        write_reports(args.output_dir)
    else:
        status["status"] = "PARTIAL"
        status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"Partial results saved under {args.output_dir}; missing {sorted(complete_keys - set(status['model_results']))}")
        return
    status["status"] = "COMPLETE"
    status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"Results saved under {args.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/npo_objective.py
"""Minimal NPO math shared by training and the CPU objective-equivalence check."""

import torch
import torch.nn.functional as F


IGNORE_INDEX = -100


def sequence_nll(logits: torch.Tensor, labels: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Return summed answer NLL and scored-token count per sequence."""
    shift_logits = logits[..., :-1, :].contiguous().float()
    shift_labels = labels[..., 1:].contiguous().to(logits.device)
    losses = F.cross_entropy(
        shift_logits.reshape(-1, shift_logits.shape[-1]),
        shift_labels.reshape(-1),
        ignore_index=IGNORE_INDEX,
        reduction="none",
    ).reshape(shift_labels.shape)
    counts = (shift_labels != IGNORE_INDEX).sum(dim=-1)
    if torch.any(counts == 0):
        raise ValueError("An NPO example has no assistant answer tokens")
    return losses.sum(dim=-1), counts


def objective_from_nlls(current_nll: torch.Tensor, reference_nll: torch.Tensor, beta: float) -> torch.Tensor:
    """Pinned Open-Unlearning NPO loss for sequence-summed answer NLL."""
    # Upstream computes -2/beta * log(sigmoid(-beta * log(pi_theta/pi_ref))).
    # Since log(pi_theta/pi_ref) = reference_nll - current_nll, this is
    # beta * (current_nll - reference_nll) inside the sigmoid.
    return (-2.0 / beta * F.logsigmoid(beta * (current_nll - reference_nll))).mean()


In [ ]:
%%writefile /kaggle/working/verify_npo_objective.py
#!/usr/bin/env python3
"""Check NPO loss/value/gradient equivalence against pinned Open-Unlearning."""

from __future__ import annotations

import argparse
import copy
import hashlib
import importlib.util
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import torch
import torch.nn.functional as F

from npo_objective import IGNORE_INDEX, objective_from_nlls, sequence_nll


class TinyCausalLM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(13, 7)
        self.head = torch.nn.Linear(7, 13, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None):
        logits = self.head(self.embedding(input_ids))
        shift_logits = logits[..., :-1, :].contiguous().float()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=IGNORE_INDEX,
            reduction="mean",
        )
        return SimpleNamespace(logits=logits, loss=loss)


def max_abs_grad_difference(left, right):
    deltas = []
    for p_left, p_right in zip(left.parameters(), right.parameters()):
        if p_left.grad is None or p_right.grad is None:
            if p_left.grad is not p_right.grad:
                return float("inf")
            continue
        deltas.append(float((p_left.grad - p_right.grad).abs().max()))
    return max(deltas, default=0.0)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--upstream-src", type=Path, required=True)
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise RuntimeError("NPO pilot protocol is not registered")
    if protocol.get("objective", {}).get("source") != "locuslab/open-unlearning@2d98027677f3468d52b6baa3fc4f2b9f2dc08820":
        raise RuntimeError("Protocol does not pin the expected Open-Unlearning implementation")
    upstream_src = args.upstream_src.resolve()
    upstream_utils = upstream_src / "trainer" / "utils.py"
    if not upstream_utils.is_file():
        raise FileNotFoundError(upstream_utils)
    spec = importlib.util.spec_from_file_location("pinned_open_unlearning_utils", upstream_utils)
    upstream = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(upstream)
    compute_dpo_loss = upstream.compute_dpo_loss

    torch.manual_seed(42)
    beta = float(protocol["objective"]["beta"])
    inputs = {
        "input_ids": torch.tensor([[1, 2, 3, 4, 5], [2, 3, 4, 6, 7]]),
        "attention_mask": torch.ones((2, 5), dtype=torch.long),
        "labels": torch.tensor([[-100, -100, 3, 4, 5], [-100, 3, 4, 6, 7]]),
    }
    base = TinyCausalLM()
    model_upstream = copy.deepcopy(base)
    model_candidate = copy.deepcopy(base)
    ref_upstream = copy.deepcopy(base)
    ref_candidate = copy.deepcopy(base)

    upstream_loss, _ = compute_dpo_loss(
        model=model_upstream,
        ref_model=ref_upstream,
        win_inputs=None,
        lose_inputs=inputs,
        beta=beta,
    )
    upstream_loss.backward()

    current_logits = model_candidate(**inputs).logits
    current_nll, _ = sequence_nll(current_logits, inputs["labels"])
    with torch.no_grad():
        reference_logits = ref_candidate(**inputs).logits
        reference_nll, _ = sequence_nll(reference_logits, inputs["labels"])
    candidate_loss = objective_from_nlls(current_nll, reference_nll, beta)
    candidate_loss.backward()

    value_diff = abs(float(upstream_loss.detach()) - float(candidate_loss.detach()))
    grad_diff = max_abs_grad_difference(model_upstream, model_candidate)
    retain_reference_loss = model_candidate(**inputs).loss.detach()
    candidate_retain_loss = current_nll.detach().sum() / (inputs["labels"][..., 1:] != IGNORE_INDEX).sum()
    retain_diff = abs(float(retain_reference_loss) - float(candidate_retain_loss))

    model_combined = copy.deepcopy(base)
    reference_combined = copy.deepcopy(base)
    combined_forget, _ = compute_dpo_loss(
        model=model_combined,
        ref_model=reference_combined,
        win_inputs=None,
        lose_inputs=inputs,
        beta=beta,
    )
    combined_retain = model_combined(**inputs).loss
    (combined_forget + combined_retain).backward()

    model_sequential = copy.deepcopy(base)
    reference_sequential = copy.deepcopy(base)
    sequential_forget, _ = compute_dpo_loss(
        model=model_sequential,
        ref_model=reference_sequential,
        win_inputs=None,
        lose_inputs=inputs,
        beta=beta,
    )
    sequential_forget.backward()
    model_sequential(**inputs).loss.backward()
    sequential_grad_diff = max_abs_grad_difference(model_combined, model_sequential)
    result = {
        "status": "PASS" if value_diff <= 1e-6 and grad_diff <= 1e-6 and retain_diff <= 1e-6 and sequential_grad_diff <= 1e-6 else "FAIL",
        "upstream": protocol["objective"]["source"],
        "upstream_utils_sha256": hashlib.sha256(upstream_utils.read_bytes()).hexdigest(),
        "candidate_objective_sha256": hashlib.sha256(Path(__file__).with_name("npo_objective.py").read_bytes()).hexdigest(),
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "python_version": sys.version,
        "torch_version": torch.__version__,
        "npo_beta": beta,
        "upstream_loss": float(upstream_loss.detach()),
        "candidate_loss": float(candidate_loss.detach()),
        "absolute_loss_difference": value_diff,
        "maximum_parameter_gradient_difference": grad_diff,
        "retain_nll_loss_difference": retain_diff,
        "combined_vs_sequential_backward_max_gradient_difference": sequential_grad_diff,
        "examples": int(inputs["input_ids"].shape[0]),
        "loss_reduction": "sum token NLL per sequence; batch mean for NPO; token mean for retain NLL",
    }
    encoded = json.dumps(result, indent=2) + "\n"
    if args.output:
        args.output.parent.mkdir(parents=True, exist_ok=True)
        args.output.write_text(encoded, encoding="utf-8")
    print(encoded, end="")
    if result["status"] != "PASS":
        raise SystemExit(1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/npo_lora_train.py
#!/usr/bin/env python3
"""Resource-bounded LoRA NPO for the pinned Open-Unlearning TOFU checkpoints.

The NPO and retain objectives follow Open-Unlearning at the pinned revision.
Only the LoRA adapter is trained; this is explicitly an adapter-based NPO
variant, not a reproduction of full-parameter NPO.
"""

from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

from tofu_official_eval import (
    CHECKPOINTS,
    DATASET_ID,
    DATASET_REVISION,
    UPSTREAM_REVISION,
    collate_scoring,
    encode_answer,
    get_bytes,
)
from npo_objective import objective_from_nlls, sequence_nll


METHOD = {
    "method": "NPO + retain NLL (LoRA adapter)",
    "beta": 0.1,
    "gamma_forget": 1.0,
    "alpha_retain": 1.0,
    "retain_loss_type": "NLL",
    "learning_rate": 1e-5,
    "weight_decay": 0.0,
    "optimizer_betas": [0.9, 0.999],
    "optimizer_eps": 1e-8,
    "epochs": 10,
    "max_length": 512,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "gradient_clip_norm": 1.0,
    "batching": "2 forget + 2 retain examples per microbatch; effective paired batch 8",
    "optimizer": "torch AdamW, betas=(0.9,0.999), eps=1e-8",
    "scheduler": "linear; one epoch warmup",
    "precision": "FP16 frozen base checkpoint; FP32 LoRA parameters and loss; SDPA",
    "retain_sampling": "uniform with replacement, one retain row per forget row, as in pinned upstream anchor=forget wrapper",
}


MODEL_FAMILIES = {
    "3b": ("llama_3_2_3b_full", 2, 4),
    "8b": ("llama_3_1_8b_full", 1, 8),
}


def fetch_jsonl(filename: str) -> tuple[list[dict], str]:
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
    raw = get_bytes(url)
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    if not rows or any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
        raise ValueError(f"Invalid pinned TOFU train file: {filename}")
    return rows, hashlib.sha256(raw).hexdigest()


def collate(rows: list[tuple[list[int], list[int]]], pad_id: int, device: torch.device):
    input_ids, attention, labels = collate_scoring(rows, pad_id)
    return input_ids.to(device), attention.to(device), labels.to(device)


def encode_rows(tokenizer, rows: list[dict]) -> list[tuple[list[int], list[int]]]:
    encoded = [encode_answer(tokenizer, row["question"], row["answer"]) for row in rows]
    if any(len(ids) > METHOD["max_length"] for ids, _ in encoded):
        raise ValueError("Found a training sequence longer than the pinned 512-token limit; no silent truncation is allowed")
    return encoded


def reference_nlls(model, encoded, pad_id: int, device: torch.device, batch_size: int = 1) -> list[float]:
    model.eval()
    values: list[float] = []
    with torch.inference_mode():
        for start in range(0, len(encoded), batch_size):
            ids, attention, labels = collate(encoded[start:start + batch_size], pad_id, device)
            outputs = model(input_ids=ids, attention_mask=attention, use_cache=False)
            nll, _ = sequence_nll(outputs.logits, labels)
            values.extend(float(x) for x in nll.cpu().tolist())
    if len(values) != len(encoded) or not all(math.isfinite(x) and x >= 0 for x in values):
        raise RuntimeError("Reference NLL precomputation was incomplete or non-finite")
    return values


def train(args) -> dict:
    global METHOD
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    objective_check = json.loads(args.objective_check.read_text(encoding="utf-8"))
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise RuntimeError("NPO protocol is not marked preregistered; refusing to train")
    if objective_check.get("status") != "PASS":
        raise RuntimeError("Pinned-objective loss/gradient equivalence did not pass")
    expected_packages = {"transformers": "5.22.1", "accelerate": "1.15.0", "peft": "0.21.2", "scipy": "1.16.3"}
    actual_packages = {name: importlib.metadata.version(name) for name in expected_packages}
    if actual_packages != expected_packages:
        raise RuntimeError(f"Training package versions differ from the verified stack: {actual_packages}")
    torchao_spec = importlib.util.find_spec("torchao")
    if torchao_spec is not None:
        try:
            torchao_version = importlib.metadata.version("torchao")
        except importlib.metadata.PackageNotFoundError:
            torchao_version = "unknown-version"
        raise RuntimeError(
            f"Optional torchao {torchao_version} is installed. This unquantized LoRA run does not use torchao; "
            "remove it before loading the base model to keep PEFT on the ordinary Linear dispatcher."
        )
    recovery_protocol = protocol.get("protocol_id") == "tofu_3b_npo_lora_profile_recovery_v1"
    if recovery_protocol:
        if args.cohort not in {"dev", "final"}:
            raise RuntimeError("Recovery training requires an explicit --cohort dev|final")
        training = protocol["training"]
        allowed_alpha = protocol["selection"]["candidate_retain_nll_weights_ascending"]
        if args.alpha_retain is None or float(args.alpha_retain) not in allowed_alpha:
            raise RuntimeError(f"--alpha-retain must be one of the preregistered candidates {allowed_alpha}")
        METHOD = dict(METHOD)
        METHOD.update({
            "beta": float(protocol["objective"]["beta"]),
            "gamma_forget": float(protocol["objective"]["forget_weight_gamma"]),
            "alpha_retain": float(args.alpha_retain),
            "learning_rate": float(training["learning_rate"]),
            "weight_decay": float(training["weight_decay"]),
            "optimizer_betas": list(training["optimizer_betas"]),
            "optimizer_eps": float(training["optimizer_eps"]),
            "scheduler": training["scheduler"],
            "precision": training["precision"],
            "epochs": int(training["epochs"]),
            "max_length": int(training["max_length"]),
            "lora_rank": int(training["lora_rank"]),
            "lora_alpha": int(training["lora_alpha"]),
            "lora_dropout": float(training["lora_dropout"]),
            "lora_target_modules": list(training["target_modules"]),
            "gradient_clip_norm": float(training["gradient_clip_norm"]),
            "batching": "2 forget + 2 retain examples per microbatch; effective paired batch 8",
            "retain_sampling": "uniform with replacement from optimizer-only retain profiles; seed 42",
        })
        expected_model_family = protocol["model"]["model_family"]
        expected_seed = int(protocol["model"]["seed"])
        locked_checkpoint = protocol["model"]
    else:
        if args.cohort is not None or args.alpha_retain is not None:
            raise RuntimeError("--cohort/--alpha-retain are only valid for the preregistered recovery protocol")
        expected_model_family = protocol.get("model_family")
        expected_seed = protocol.get("seed")
        locked_checkpoint = protocol.get("base_checkpoint", {})

    if args.model != expected_model_family or args.seed != expected_seed:
        raise RuntimeError("Requested model/seed does not match the locked NPO diagnostic protocol")
    model_key, microbatch, accumulation = MODEL_FAMILIES[args.model]
    spec = CHECKPOINTS[model_key]
    if (spec.get("repo_id"), spec.get("revision")) != (locked_checkpoint.get("repo_id"), locked_checkpoint.get("revision")):
        raise RuntimeError("Trainer checkpoint/revision differs from the preregistered protocol")
    if objective_check.get("upstream") != f"locuslab/open-unlearning@{UPSTREAM_REVISION}":
        raise RuntimeError("Objective receipt is for the wrong Open-Unlearning revision")
    if float(objective_check.get("npo_beta", -1)) != float(protocol["objective"]["beta"]):
        raise RuntimeError("Objective check beta differs from the preregistered protocol")
    if recovery_protocol:
        locked_training = protocol["training"]
        expected_modules = locked_training["target_modules"]
        expected_precision = locked_training["precision"]
        expected_scheduler = locked_training["scheduler"]
        expected_effective_batch = locked_training["effective_paired_batch_size"]
        expected_clip = locked_training["gradient_clip_norm"]
        expected_retain_loss = "NLL"
    else:
        locked_training = protocol["adaptation"]
        expected_modules = locked_training["target_modules"]
        expected_precision = locked_training["precision"]
        expected_scheduler = locked_training["scheduler"]
        expected_effective_batch = locked_training["effective_paired_batch_size"]
        expected_clip = locked_training["gradient_clip_norm"]
        expected_retain_loss = "NLL"
    if (
        METHOD["lora_rank"] != locked_training["lora_rank"]
        or METHOD["lora_alpha"] != locked_training["lora_alpha"]
        or METHOD["lora_dropout"] != locked_training["lora_dropout"]
        or METHOD["lora_target_modules"] != expected_modules
        or METHOD["learning_rate"] != locked_training["learning_rate"]
        or METHOD["weight_decay"] != (float(locked_training["weight_decay"]) if recovery_protocol else float(locked_training["optimizer"].split("weight_decay=")[-1].rstrip(")")))
        or (recovery_protocol and METHOD["optimizer_betas"] != locked_training["optimizer_betas"])
        or (recovery_protocol and METHOD["optimizer_eps"] != locked_training["optimizer_eps"])
        or METHOD["beta"] != protocol["objective"]["beta"]
        or METHOD["gamma_forget"] != protocol["objective"]["forget_weight_gamma"]
        or (not recovery_protocol and METHOD["alpha_retain"] != protocol["objective"]["retain_weight_alpha"])
        or METHOD["retain_loss_type"] != expected_retain_loss
        or METHOD["precision"] != expected_precision
        or METHOD["scheduler"] != expected_scheduler
        or METHOD["epochs"] != locked_training["epochs"]
        or METHOD["max_length"] != locked_training["max_length"]
        or METHOD["gradient_clip_norm"] != expected_clip
        or microbatch * accumulation != expected_effective_batch
    ):
        raise RuntimeError("Trainer hyperparameters differ from the preregistered NPO-LoRA protocol")
    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
        raise RuntimeError("NPO requires two GPUs; stopped before model download")
    gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    if gpu_names.count("Tesla T4") != 2:
        raise RuntimeError(f"Protocol requires Kaggle T4 x2, found {gpu_names}")

    output_dir: Path = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=False)
    run_type = "bounded_real_data_smoke" if args.smoke else "exploratory_single_seed"
    started = time.monotonic()
    receipt = {
        "status": "RUNNING",
        "run_type": run_type,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": spec,
        "dataset": {"repo": DATASET_ID, "revision": DATASET_REVISION},
        "upstream_code_revision": UPSTREAM_REVISION,
        "method": METHOD,
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "objective_equivalence": objective_check,
        "training_script_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
        "seed": args.seed,
        "gpu_names": gpu_names,
        "package_versions": {
            "torch": torch.__version__,
            "transformers": importlib.metadata.version("transformers"),
            "accelerate": importlib.metadata.version("accelerate"),
            "peft": importlib.metadata.version("peft"),
            "scipy": importlib.metadata.version("scipy"),
            "torchao": None,
        },
        "gpu_total_memory_bytes": [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())],
        "smoke_rows": 8 if args.smoke else None,
        "training_started": False,
        "adapter_saved": False,
    }
    if recovery_protocol:
        receipt["followup"] = {
            "protocol_id": protocol["protocol_id"],
            "cohort": args.cohort,
            "selected_alpha_retain": float(args.alpha_retain),
            "study_type": protocol["study_type"],
        }
    receipt_path = output_dir / "training_receipt.json"
    trace_path = output_dir / "loss_trace.jsonl"
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")

    try:
        forget_rows, forget_sha = fetch_jsonl("forget10.json")
        retain_rows, retain_sha = fetch_jsonl("retain90.json")
        if len(forget_rows) != 400 or len(retain_rows) != 3600:
            raise ValueError(f"Unexpected pinned train counts: forget={len(forget_rows)}, retain={len(retain_rows)}")
        if recovery_protocol:
            expected_hashes = protocol["data"]["sha256"]
            if forget_sha != expected_hashes["forget10.json"] or retain_sha != expected_hashes["retain90.json"]:
                raise ValueError("Fetched training data hash differs from the preregistered recovery protocol")
            block_size = int(protocol["data"]["profile_block_size"])
            assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
            forget_key = f"forget10_{args.cohort}_train_and_eval"
            if forget_key not in assignments:
                raise ValueError(f"No forget profile assignment for cohort {args.cohort}")
            forget_blocks = [b for first, last in assignments[forget_key] for b in range(first, last + 1)]
            retain_blocks = [b for first, last in assignments["retain90_optimizer_train"] for b in range(first, last + 1)]
            if len(forget_blocks) != len(set(forget_blocks)) or len(retain_blocks) != len(set(retain_blocks)):
                raise ValueError("Recovery training profile assignments contain duplicates")
            forget_source_indices = [i for i in range(len(forget_rows)) if i // block_size in forget_blocks]
            retain_source_indices = [i for i in range(len(retain_rows)) if i // block_size in retain_blocks]
            forget_rows = [forget_rows[i] for i in forget_source_indices]
            retain_rows = [retain_rows[i] for i in retain_source_indices]
            if len(forget_rows) != 200 or len(retain_rows) != 2760:
                raise ValueError(f"Wrong {args.cohort} cohort size: forget={len(forget_rows)}, retain={len(retain_rows)}")
            receipt["dataset"].update({
                "forget10_source_rows_selected": forget_source_indices,
                "forget10_profile_blocks_selected": forget_blocks,
                "retain90_source_rows_selected": retain_source_indices,
                "retain90_profile_blocks_selected": retain_blocks,
                "forget10_rows_selected": len(forget_rows),
                "retain90_rows_selected": len(retain_rows),
            })
        else:
            locked_files = protocol["dataset"]["training"]
            if forget_sha != locked_files["forget_file"]["sha256"] or retain_sha != locked_files["retain_file"]["sha256"]:
                raise ValueError("Fetched training data hash differs from the preregistered dataset revision and hash")
        if not recovery_protocol and (len(forget_rows) != 400 or len(retain_rows) != 3600):
            raise ValueError("Fetched training data hash differs from the preregistered dataset revision and hash")
        receipt["dataset"].update({
            "forget10_rows": len(forget_rows),
            "retain90_rows": len(retain_rows),
            "forget10_sha256": forget_sha,
            "retain90_sha256": retain_sha,
        })

        cache_root = Path(os.environ.get("TOFU_MODEL_CACHE_ROOT", "/kaggle/working/model_cache"))
        checkpoint_cache = cache_root / model_key
        tokenizer = AutoTokenizer.from_pretrained(
            spec["repo_id"], revision=spec["revision"], cache_dir=str(checkpoint_cache), use_fast=True
        )
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        if not tokenizer.is_fast:
            raise RuntimeError("Pinned tokenizer must be fast")
        forget_encoded = encode_rows(tokenizer, forget_rows)
        retain_encoded = encode_rows(tokenizer, retain_rows)
        receipt["tokenization"] = {
            "max_length_observed": max(len(x[0]) for x in forget_encoded + retain_encoded),
            "truncated_examples": 0,
            "prompt_answer_prefix_check": "PASS for every sequence",
        }

        torch.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = False
        model = AutoModelForCausalLM.from_pretrained(
            spec["repo_id"],
            revision=spec["revision"],
            cache_dir=str(checkpoint_cache),
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="balanced",
            max_memory={0: "13GiB", 1: "13GiB", "cpu": "40GiB"},
            attn_implementation="sdpa",
        )
        device_map = getattr(model, "hf_device_map", {})
        offload = sorted({str(v) for v in device_map.values() if str(v).lower() in {"cpu", "disk"}})
        if offload:
            raise RuntimeError(f"Refusing training with CPU/disk offload: {offload}")
        input_device = model.get_input_embeddings().weight.device
        receipt["runtime"] = {
            "device_map": {str(k): str(v) for k, v in device_map.items()},
            "cpu_or_disk_offload": False,
            "input_device": str(input_device),
            "dtype": "float16",
            "attention": "sdpa",
        }

        if args.smoke:
            forget_encoded = forget_encoded[:8]
            # Keep the full retain pool in the real-data smoke; sample only the
            # first eight target steps to bound runtime and avoid a scored run.
            steps_to_run = 1
            batch_size = 1
            accumulation_steps = 1
            epochs = 1
        else:
            steps_to_run = None
            batch_size = microbatch
            accumulation_steps = accumulation
            epochs = METHOD["epochs"]

        effective_forget = 8
        batches_per_epoch = math.ceil(len(forget_encoded) / batch_size)
        steps_per_epoch = math.ceil(batches_per_epoch / accumulation_steps)
        planned_steps = 1 if args.smoke else steps_per_epoch * epochs
        planned_trace_rows = min(batches_per_epoch * epochs, planned_steps * accumulation_steps)
        receipt["training_schedule"] = {
            "epochs": epochs,
            "forget_examples_per_microbatch": batch_size,
            "retain_examples_per_microbatch": batch_size,
            "gradient_accumulation_steps": accumulation_steps,
            "forget_examples_per_optimizer_step": batch_size * accumulation_steps,
            "microbatches_per_epoch": batches_per_epoch,
            "optimizer_steps_per_epoch": steps_per_epoch,
            "planned_optimizer_steps": planned_steps,
            "planned_loss_trace_rows": planned_trace_rows,
            "loss_trace_unit": "one mean-loss record per paired microbatch",
        }
        receipt["training_started"] = True
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        ref_nll = reference_nlls(model, forget_encoded, int(tokenizer.pad_token_id), input_device)

        model.config.use_cache = False
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        lora = LoraConfig(
            r=METHOD["lora_rank"],
            lora_alpha=METHOD["lora_alpha"],
            lora_dropout=METHOD["lora_dropout"],
            target_modules=METHOD["lora_target_modules"],
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora)
        trainable = [p for p in model.parameters() if p.requires_grad]
        if not trainable or any(p.dtype != torch.float32 for p in trainable):
            raise RuntimeError("Expected nonempty FP32 LoRA-only trainable parameters")
        for name, parameter in model.named_parameters():
            if parameter.requires_grad and "lora_" not in name:
                raise RuntimeError(f"Unexpected trainable base parameter: {name}")
        model.train()
        optimizer = torch.optim.AdamW(
            trainable,
            lr=METHOD["learning_rate"],
            betas=tuple(METHOD["optimizer_betas"]),
            eps=METHOD["optimizer_eps"],
            weight_decay=METHOD["weight_decay"],
        )

        if not args.smoke:
            total_steps = planned_steps
            warmup_steps = steps_per_epoch
        else:
            total_steps, warmup_steps = planned_steps, 0
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        rng = random.Random(args.seed)
        global_step = 0
        # PEFT initializes LoRA-A with zero gradient on the first step and
        # LoRA-B at zero; inspect every adapter tensor so the smoke checks an
        # actual update rather than whichever tensor happens to be first.
        smoke_initial = [p.detach().float().cpu().clone() for p in trainable] if args.smoke else None
        optimizer.zero_grad(set_to_none=True)
        with trace_path.open("w", encoding="utf-8") as trace:
            for epoch in range(epochs):
                order = list(range(len(forget_encoded)))
                rng.shuffle(order)
                micro_in_update = 0
                pending_losses = []
                for start in range(0, len(order), batch_size):
                    indices = order[start:start + batch_size]
                    if not indices:
                        continue
                    f_rows = [forget_encoded[i] for i in indices]
                    retain_indices = [rng.randrange(len(retain_encoded)) for _ in indices]
                    r_rows = [retain_encoded[i] for i in retain_indices]
                    f_ids, f_mask, f_labels = collate(f_rows, int(tokenizer.pad_token_id), input_device)
                    r_ids, r_mask, r_labels = collate(r_rows, int(tokenizer.pad_token_id), input_device)
                    f_logits = model(input_ids=f_ids, attention_mask=f_mask, use_cache=False).logits
                    f_current, _ = sequence_nll(f_logits, f_labels)
                    refs = torch.tensor([ref_nll[i] for i in indices], dtype=f_current.dtype, device=f_current.device)
                    forget_loss = objective_from_nlls(f_current, refs, METHOD["beta"])
                    if not torch.isfinite(forget_loss):
                        raise FloatingPointError("Non-finite NPO forget loss")
                    forget_loss_value = float(forget_loss.detach().cpu())
                    (METHOD["gamma_forget"] * forget_loss / accumulation_steps).backward()
                    del f_logits, f_current, refs, forget_loss

                    r_logits = model(input_ids=r_ids, attention_mask=r_mask, use_cache=False).logits
                    r_nll, r_counts = sequence_nll(r_logits, r_labels)
                    retain_loss = (r_nll.sum() / r_counts.sum().to(r_nll.dtype))
                    if not torch.isfinite(retain_loss):
                        raise FloatingPointError("Non-finite retain NLL")
                    retain_loss_value = float(retain_loss.detach().cpu())
                    (METHOD["alpha_retain"] * retain_loss / accumulation_steps).backward()
                    del r_logits, r_nll, r_counts, retain_loss
                    combined_loss_value = (
                        METHOD["gamma_forget"] * forget_loss_value
                        + METHOD["alpha_retain"] * retain_loss_value
                    )
                    if not math.isfinite(combined_loss_value):
                        raise FloatingPointError("Non-finite combined NPO+retain loss")
                    micro_in_update += 1
                    pending_losses.append({
                        "epoch": epoch + 1,
                        "forget_npo_loss": forget_loss_value,
                        "retain_token_nll": retain_loss_value,
                        "combined_loss": combined_loss_value,
                        "forget_examples_in_microbatch": len(indices),
                        "retain_examples_in_microbatch": len(retain_indices),
                    })

                    final_micro = start + batch_size >= len(order)
                    if micro_in_update == accumulation_steps or final_micro:
                        # Correct for a short final accumulation window.
                        if final_micro and micro_in_update < accumulation_steps:
                            scale = accumulation_steps / micro_in_update
                            for parameter in trainable:
                                if parameter.grad is not None:
                                    parameter.grad.mul_(scale)
                        grad_norm = torch.nn.utils.clip_grad_norm_(trainable, METHOD["gradient_clip_norm"])
                        if not torch.isfinite(grad_norm):
                            raise FloatingPointError("Non-finite LoRA gradient norm")
                        optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad(set_to_none=True)
                        global_step += 1
                        for record in pending_losses:
                            record.update({
                                "global_step": global_step,
                                "learning_rate": scheduler.get_last_lr()[0],
                                "gradient_norm": float(grad_norm.detach().cpu()),
                            })
                            trace.write(json.dumps(record) + "\n")
                        trace.flush()
                        pending_losses.clear()
                        micro_in_update = 0
                        if steps_to_run is not None and global_step >= steps_to_run:
                            break
                if steps_to_run is not None and global_step >= steps_to_run:
                    break

        if global_step == 0:
            raise RuntimeError("No optimizer update was performed")
        smoke_parameter_delta = None
        if args.smoke:
            smoke_parameter_delta = max(
                float((parameter.detach().float().cpu() - initial).abs().max())
                for parameter, initial in zip(trainable, smoke_initial)
            )
            if not math.isfinite(smoke_parameter_delta) or smoke_parameter_delta <= 0:
                raise RuntimeError("Real-data smoke loss was finite but no LoRA parameter changed")
        receipt.update({
            "status": "SMOKE_PASS" if args.smoke else "TRAINING_COMPLETE",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "global_steps": global_step,
            "trainable_parameters": sum(p.numel() for p in trainable),
            "all_parameters": sum(p.numel() for p in model.parameters()),
            "trainable_parameter_fraction": sum(p.numel() for p in trainable) / sum(p.numel() for p in model.parameters()),
            "optimizer_steps_with_finite_loss_and_gradient": global_step,
            "smoke_max_abs_lora_parameter_update": smoke_parameter_delta,
            "runtime_seconds": round(time.monotonic() - started, 2),
        })
        if not args.smoke:
            model.save_pretrained(output_dir / "adapter", safe_serialization=True)
            tokenizer.save_pretrained(output_dir / "adapter")
            receipt["adapter_saved"] = True
            receipt["adapter_files_sha256"] = {
                p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                for p in sorted((output_dir / "adapter").iterdir()) if p.is_file()
            }
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        print(json.dumps(receipt, indent=2), flush=True)
        return receipt
    except Exception as error:
        receipt.update({
            "status": "ERROR",
            "error": f"{type(error).__name__}: {error}",
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "runtime_seconds": round(time.monotonic() - started, 2),
        })
        receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
        raise


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", choices=MODEL_FAMILIES, required=True)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--objective-check", type=Path, required=True)
    parser.add_argument("--cohort", choices=("dev", "final"))
    parser.add_argument("--alpha-retain", type=float)
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    train(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/validate_npo_recovery_protocol.py
#!/usr/bin/env python3
"""Validate the locked TOFU profile-cohort recovery design and source data."""

from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
from urllib.request import Request, urlopen


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"


def get_bytes(url: str) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-cohort-audit/1.0"})
    with urlopen(request, timeout=60) as response:
        return response.read()


def fetch(protocol: dict, filename: str) -> tuple[list[dict], str]:
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{filename}"
    raw = get_bytes(url)
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    return rows, hashlib.sha256(raw).hexdigest()


def expand(ranges: list[list[int]]) -> list[int]:
    blocks = [block for first, last in ranges for block in range(first, last + 1)]
    if len(blocks) != len(set(blocks)):
        raise ValueError("A profile assignment range contains duplicate blocks")
    return blocks


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--protocol", type=Path, default=Path(__file__).with_name("npo_lora_recovery_protocol.json"))
    parser.add_argument("--output", type=Path)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise ValueError("Recovery protocol is not in its required locked-before-GPU status")
    if protocol["data"]["repo_id"] != DATASET_ID or protocol["data"]["revision"] != DATASET_REVISION:
        raise ValueError("Protocol data source does not match the pinned TOFU evaluator")
    size = int(protocol["data"]["profile_block_size"])
    rows_by_file = {}
    observed_hashes = {}
    for filename, expected_count in protocol["data"]["expected_rows"].items():
        rows, digest = fetch(protocol, filename)
        if len(rows) != expected_count:
            raise ValueError(f"{filename}: expected {expected_count} rows, got {len(rows)}")
        if digest != protocol["data"]["sha256"][filename]:
            raise ValueError(f"{filename}: SHA-256 mismatch")
        if len(rows) % size:
            raise ValueError(f"{filename}: rows do not form complete {size}-QA profiles")
        rows_by_file[filename] = rows
        observed_hashes[filename] = digest

    assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
    forget_dev = expand(assignments["forget10_dev_train_and_eval"])
    forget_final = expand(assignments["forget10_final_train_and_eval"])
    retain_train = expand(assignments["retain90_optimizer_train"])
    retain_dev = expand(assignments["retain90_dev_utility_eval"])
    retain_final = expand(assignments["retain90_final_utility_eval"])
    holdout_dev = expand(assignments["holdout10_dev_utility_eval"])
    holdout_final = expand(assignments["holdout10_final_utility_eval"])
    retain_excluded = set(assignments["retain90_excluded_conflicting_question_profiles"])

    if set(forget_dev) & set(forget_final) or sorted(forget_dev + forget_final) != list(range(20)):
        raise ValueError("Forget10 dev/final assignment must partition profile blocks 0..19 exactly")
    if set(retain_train) & set(retain_dev) or set(retain_train) & set(retain_final) or set(retain_dev) & set(retain_final):
        raise ValueError("Retain optimizer, dev, and final profile assignments overlap")
    if retain_excluded & (set(retain_train) | set(retain_dev) | set(retain_final)):
        raise ValueError("Conflicting-question retain profiles must be excluded from all pools")
    if set(retain_train) | set(retain_dev) | set(retain_final) | retain_excluded != set(range(180)):
        raise ValueError("Retain profile assignment must account for all 180 profiles")
    if set(holdout_dev) & set(holdout_final) or sorted(holdout_dev + holdout_final) != list(range(20)):
        raise ValueError("Holdout dev/final assignment must partition profile blocks 0..19 exactly")

    retain_rows = rows_by_file["retain90.json"]
    answers_by_question: dict[str, set[str]] = {}
    for row in retain_rows:
        answers_by_question.setdefault(row["question"], set()).add(row["answer"])
    collision_indices = [i for i, row in enumerate(retain_rows) if len(answers_by_question[row["question"]]) > 1]
    collision_profiles = sorted({i // size for i in collision_indices})
    if collision_profiles != sorted(retain_excluded):
        raise ValueError(f"Conflicting-question profiles differ from protocol: {collision_profiles}")

    forget = rows_by_file["forget10.json"]
    forget_perturbed = rows_by_file["forget10_perturbed.json"]
    if any((a["question"], a["answer"]) != (b["question"], b["answer"]) for a, b in zip(forget, forget_perturbed)):
        raise ValueError("Forget10 perturbed rows do not align with forget10 in source order")

    counts = {
        "dev": {
            "forget10": len(forget_dev) * size,
            "retain90": len(retain_dev) * size,
            "holdout10": len(holdout_dev) * size,
            "retain_optimizer": len(retain_train) * size,
        },
        "final": {
            "forget10": len(forget_final) * size,
            "retain90": len(retain_final) * size,
            "holdout10": len(holdout_final) * size,
            "retain_optimizer": len(retain_train) * size,
        },
    }
    if counts != {"dev": {"forget10": 200, "retain90": 400, "holdout10": 200, "retain_optimizer": 2760},
                  "final": {"forget10": 200, "retain90": 400, "holdout10": 200, "retain_optimizer": 2760}}:
        raise ValueError(f"Unexpected source-cohort sizes: {counts}")

    receipt = {
        "status": "PASS_PROFILE_COHORT_DATA_AUDIT",
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "dataset_id": DATASET_ID,
        "dataset_revision": DATASET_REVISION,
        "file_sha256": observed_hashes,
        "profile_block_size": size,
        "profile_counts": {
            "forget10": len(forget) // size,
            "retain90": len(retain_rows) // size,
            "holdout10": len(rows_by_file["holdout10.json"]) // size,
        },
        "assignments": assignments,
        "conflicting_question_source_rows_excluded": collision_indices,
        "counts": counts,
        "training_started": False,
        "model_weights_loaded": False,
    }
    serialized = json.dumps(receipt, indent=2) + "\n"
    if args.output:
        args.output.write_text(serialized, encoding="utf-8")
    print(serialized, end="")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/evaluate_npo_profile_cohort.py
#!/usr/bin/env python3
"""Evaluate and gate the preregistered TOFU NPO-LoRA profile cohorts."""

from __future__ import annotations

import argparse
import hashlib
import json
import random
from pathlib import Path

from tofu_official_eval import (
    BOOTSTRAP_DRAWS,
    CHECKPOINTS,
    DATASET_REVISION,
    evaluate_checkpoint,
    load_data,
)


def expand(ranges: list[list[int]]) -> list[int]:
    return [block for first, last in ranges for block in range(first, last + 1)]


def make_rows(data: dict[str, list[dict]], split: str, profile_blocks: list[int]) -> list[dict]:
    profile_size = 20
    if split == "forget10":
        source = data["forget10.json"]
        eval_source = data["forget10_perturbed.json"]
        if len(source) != len(eval_source) or any(
            (a["question"], a["answer"]) != (b["question"], b["answer"])
            for a, b in zip(source, eval_source)
        ):
            raise ValueError("forget10_perturbed does not align with forget10 source order")
    elif split == "retain90":
        source = data["retain90.json"]
        eval_source = source
    elif split == "holdout10":
        source = data["holdout10.json"]
        eval_source = source
    else:
        raise ValueError(f"Unsupported profile split: {split}")

    selected_blocks = set(profile_blocks)
    output = []
    for source_index, row in enumerate(source):
        block = source_index // profile_size
        if block not in selected_blocks:
            continue
        paired = eval_source[source_index]
        output.append({
            "split": split,
            "eval_row_index": len(output),
            "source_row_index": source_index,
            "profile_block": block,
            "question": row["question"],
            "answer": row["answer"],
            "ambiguous_question_collision": False,
            "paraphrased_answer": paired.get("paraphrased_answer") if split == "forget10" else None,
            "perturbed_answers": paired.get("perturbed_answer") if split == "forget10" else [],
        })
    expected = len(selected_blocks) * profile_size
    if len(output) != expected:
        raise ValueError(f"{split}: expected {expected} selected rows, got {len(output)}")
    if split == "forget10" and any(not row["perturbed_answers"] for row in output):
        raise ValueError("A target forget row has no perturbed answers; Truth Ratio cannot be calculated")
    return output


def cohort_sets(data: dict[str, list[dict]], protocol: dict, cohort: str) -> dict[str, list[dict]]:
    assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
    return {
        split: make_rows(data, split, expand(assignments[f"{split}_{cohort}_utility_eval"] if split != "forget10" else assignments[f"forget10_{cohort}_train_and_eval"]))
        for split in ("forget10", "retain90", "holdout10")
    }


def reference_sets(data: dict[str, list[dict]], protocol: dict) -> dict[str, list[dict]]:
    assignments = protocol["data"]["profile_assignment_ranges_inclusive"]
    target = list(range(20))
    retain = expand(assignments["retain90_dev_utility_eval"]) + expand(assignments["retain90_final_utility_eval"])
    holdout = list(range(20))
    return {
        "forget10": make_rows(data, "forget10", target),
        "retain90": make_rows(data, "retain90", retain),
        "holdout10": make_rows(data, "holdout10", holdout),
    }


def paired_cluster_delta(base_rows: list[dict], candidate_rows: list[dict], field: str, seed: int = 42) -> dict:
    base_by_index = {int(row["source_row_index"]): row for row in base_rows}
    groups: dict[int, list[float]] = {}
    for row in candidate_rows:
        base = base_by_index.get(int(row["source_row_index"]))
        if base is None or field not in row or field not in base:
            continue
        groups.setdefault(int(row["profile_block"]), []).append(float(row[field]) - float(base[field]))
    if not groups:
        raise ValueError(f"No matched baseline/candidate rows for {field}")
    block_ids = sorted(groups)
    point = sum(sum(values) for values in groups.values()) / sum(len(values) for values in groups.values())
    rng = random.Random(seed)
    estimates = []
    for _ in range(BOOTSTRAP_DRAWS):
        selected = [rng.choice(block_ids) for _ in block_ids]
        values = [value for block in selected for value in groups[block]]
        estimates.append(sum(values) / len(values))
    estimates.sort()
    return {
        "n_questions": sum(len(values) for values in groups.values()),
        "n_profile_blocks": len(groups),
        "mean_delta_candidate_minus_full": point,
        "profile_cluster_bootstrap_95pct_ci": [
            estimates[int(0.025 * (BOOTSTRAP_DRAWS - 1))],
            estimates[int(0.975 * (BOOTSTRAP_DRAWS - 1))],
        ],
        "bootstrap_draws": BOOTSTRAP_DRAWS,
        "bootstrap_seed": seed,
    }


def gate_comparison(base: dict, candidate: dict, protocol: dict) -> dict:
    tests = {
        "forget10.truth_ratio_score": ("forget10", "truth_ratio_score", "lower_gt_zero"),
        "forget10.answer_probability": ("forget10", "answer_probability", "upper_lt_zero"),
        "retain90.answer_probability": ("retain90", "answer_probability", "lower_gt_minus_margin"),
        "retain90.rougeL_recall": ("retain90", "rougeL_recall", "lower_gt_minus_margin"),
        "holdout10.answer_probability": ("holdout10", "answer_probability", "lower_gt_minus_margin"),
    }
    margin = float(protocol["selection"]["noninferiority_margin"])
    checks = {}
    for name, (split, field, rule) in tests.items():
        delta = paired_cluster_delta(base["per_example"][split], candidate["per_example"][split], field)
        low, high = delta["profile_cluster_bootstrap_95pct_ci"]
        passed = low > 0 if rule == "lower_gt_zero" else high < 0 if rule == "upper_lt_zero" else low > -margin
        checks[name] = {
            "delta": delta["mean_delta_candidate_minus_full"],
            "ci95": [low, high],
            "rule": rule,
            "threshold": 0.0 if rule.endswith("zero") else -margin,
            "pass": bool(passed),
            "n_questions": delta["n_questions"],
            "n_profile_blocks": delta["n_profile_blocks"],
        }
    result = {
        "status": "PASS_DIAGNOSTIC_GATES" if all(item["pass"] for item in checks.values()) else "FAIL_DIAGNOSTIC_GATES",
        "interpretation": "One-seed exploratory profile-cohort follow-up; not confirmatory and not a standard full-split TOFU run.",
        "utility_and_forgetting_gates": checks,
        "all_gates_pass": all(item["pass"] for item in checks.values()),
        "noninferiority_margin": margin,
        "bootstrap": {"draws": BOOTSTRAP_DRAWS, "seed": 42, "unit": "profile block, paired by source row index"},
    }
    return result


def load_adapter_receipt(adapter_dir: Path, cohort: str, alpha: float) -> dict:
    receipt_path = adapter_dir.parent / "training_receipt.json"
    if not receipt_path.is_file():
        raise FileNotFoundError(f"Adapter training receipt missing: {receipt_path}")
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    if receipt.get("status") != "TRAINING_COMPLETE" or not receipt.get("adapter_saved"):
        raise ValueError("Adapter receipt is not a complete training run")
    followup = receipt.get("followup", {})
    if followup.get("cohort") != cohort or float(followup.get("selected_alpha_retain", -1)) != alpha:
        raise ValueError("Adapter cohort/alpha does not match evaluation request")
    actual_hashes = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(adapter_dir.iterdir()) if path.is_file()
    }
    if actual_hashes != receipt.get("adapter_files_sha256"):
        raise ValueError("Adapter files do not match their training receipt hashes")
    return receipt


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--protocol", type=Path, default=Path(__file__).with_name("npo_lora_recovery_protocol.json"))
    parser.add_argument("--mode", choices=("reference", "dev", "final"), required=True)
    parser.add_argument("--alpha-retain", type=float)
    parser.add_argument("--adapter-dir", type=Path)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    if protocol.get("protocol_id") != "tofu_3b_npo_lora_profile_recovery_v1":
        raise ValueError("Wrong profile recovery protocol")
    if protocol["status"] != "PREREGISTERED_BEFORE_GPU_RUN":
        raise ValueError("Recovery protocol is not preregistered")
    if (args.mode == "reference") != (args.adapter_dir is None):
        raise ValueError("Reference mode requires no adapter; dev/final modes require --adapter-dir")
    if args.mode != "reference" and args.alpha_retain not in protocol["selection"]["candidate_retain_nll_weights_ascending"]:
        raise ValueError("Adapter alpha is not in the locked candidate grid")

    data, hashes = load_data()
    for filename, expected_hash in protocol["data"]["sha256"].items():
        if hashes.get(filename) != expected_hash:
            raise ValueError(f"Pinned source hash mismatch: {filename}")
    evaluation_sets = reference_sets(data, protocol) if args.mode == "reference" else cohort_sets(data, protocol, args.mode)
    if args.mode == "reference":
        adapter_path = None
        result_key = "full_reference"
    else:
        load_adapter_receipt(args.adapter_dir, args.mode, float(args.alpha_retain))
        adapter_path = args.adapter_dir
        result_key = f"{args.mode}_alpha_{args.alpha_retain:g}"
    args.output_dir.mkdir(parents=True, exist_ok=True)
    result = evaluate_checkpoint(
        "llama_3_2_3b_full",
        args.output_dir,
        evaluation_sets,
        hashes,
        score_batch_size=1,
        generation_batch_size=1,
        max_memory_per_gpu_gib=12.5,
        adapter_path=adapter_path,
        result_key=result_key,
        adapter_seed=42 if adapter_path else None,
        profile_cohort_mode=True,
    )
    if result.get("status") != "PASS_PROFILE_COHORT_EVAL":
        raise RuntimeError("Profile cohort evaluation did not complete")
    if args.mode == "reference":
        return

    reference_path = args.output_dir / "full_reference_profile_eval.json"
    if not reference_path.is_file():
        raise FileNotFoundError("Full-checkpoint profile reference evaluation must be completed first")
    baseline = json.loads(reference_path.read_text(encoding="utf-8"))
    comparison = gate_comparison(baseline, result, protocol)
    comparison.update({
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "model_revision": CHECKPOINTS["llama_3_2_3b_full"]["revision"],
        "dataset_revision": DATASET_REVISION,
        "cohort": args.mode,
        "alpha_retain": float(args.alpha_retain),
        "training_receipt_sha256": hashlib.sha256((args.adapter_dir.parent / "training_receipt.json").read_bytes()).hexdigest(),
        "evaluation_receipt": f"{result_key}_profile_eval.json",
    })
    comparison_path = args.output_dir / f"{args.mode}_alpha_{args.alpha_retain:g}_comparison.json"
    comparison_path.write_text(json.dumps(comparison, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(comparison, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/run_npo_profile_recovery.py
#!/usr/bin/env python3
"""Run the locked TOFU NPO-LoRA alpha-selection and held-out profile follow-up."""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path


HERE = Path(__file__).resolve().parent


def run_stage(name: str, command: list[str], root: Path) -> int:
    log = root / "logs" / f"{name}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {name} ===", flush=True)
    print("COMMAND:", " ".join(command), flush=True)
    start = time.monotonic()
    with log.open("w", encoding="utf-8") as sink:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line.rstrip("\n"), flush=True)
            sink.write(line)
            sink.flush()
        return_code = process.wait()
    print(f"STAGE {name} exit={return_code}; elapsed_seconds={time.monotonic() - start:.1f}", flush=True)
    return return_code


def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")


def archive(root: Path, destination: Path) -> dict:
    if destination.exists():
        destination.unlink()
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for path in sorted(root.rglob("*")):
            if path.is_file() and path != destination:
                zf.write(path, path.relative_to(root.parent))
    digest = hashlib.sha256(destination.read_bytes()).hexdigest()
    return {"path": str(destination), "bytes": destination.stat().st_size, "sha256": digest}


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", type=Path, default=Path("/kaggle/working/tofu_npo_profile_recovery"))
    parser.add_argument("--protocol", type=Path, default=HERE / "npo_lora_recovery_protocol.json")
    parser.add_argument("--upstream-src", type=Path, default=Path("/kaggle/working/open-unlearning/src"))
    parser.add_argument("--objective-check", type=Path, default=Path("/kaggle/working/npo_recovery_objective_check.json"))
    parser.add_argument("--artifact-zip", type=Path, default=Path("/kaggle/working/tofu_npo_profile_recovery_artifacts.zip"))
    args = parser.parse_args()
    root = args.root
    root.mkdir(parents=True, exist_ok=False)
    for path in (
        HERE / "npo_lora_recovery_protocol.json",
        HERE / "validate_npo_recovery_protocol.py",
        HERE / "npo_lora_train.py",
        HERE / "npo_objective.py",
        HERE / "verify_npo_objective.py",
        HERE / "tofu_official_eval.py",
        HERE / "evaluate_npo_profile_cohort.py",
        HERE / "requirements-npo-pilot.txt",
    ):
        shutil.copy2(path, root / path.name)
    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))
    if protocol.get("status") != "PREREGISTERED_BEFORE_GPU_RUN":
        raise RuntimeError("Recovery protocol is not locked before GPU training")
    if not os.environ.get("CUDA_VISIBLE_DEVICES", "") and not __import__("torch").cuda.is_available():
        raise RuntimeError("CUDA is unavailable")
    import torch
    gpu_names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
    if len(gpu_names) != 2 or gpu_names.count("Tesla T4") != 2:
        raise RuntimeError(f"Recovery protocol requires exactly two Tesla T4 GPUs, found {gpu_names}")

    receipt = {
        "status": "RUNNING",
        "protocol_id": protocol["protocol_id"],
        "protocol_sha256": hashlib.sha256(args.protocol.read_bytes()).hexdigest(),
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "gpu_names": gpu_names,
        "training_started": False,
        "candidate_attempts": [],
        "selected_alpha_retain": None,
        "final_training_started": False,
    }
    receipt_path = root / "recovery_run_receipt.json"
    write_json(receipt_path, receipt)
    try:
        audit_path = root / "profile_cohort_data_audit.json"
        command = [sys.executable, str(HERE / "validate_npo_recovery_protocol.py"), "--protocol", str(args.protocol), "--output", str(audit_path)]
        if run_stage("data-and-profile-audit", command, root):
            raise RuntimeError("Pinned source data/profile cohort audit failed")
        audit = json.loads(audit_path.read_text(encoding="utf-8"))
        if audit.get("status") != "PASS_PROFILE_COHORT_DATA_AUDIT":
            raise RuntimeError("Profile cohort audit did not pass")

        command = [sys.executable, str(HERE / "verify_npo_objective.py"), "--upstream-src", str(args.upstream_src), "--protocol", str(args.protocol), "--output", str(args.objective_check)]
        if run_stage("objective-equivalence", command, root):
            raise RuntimeError("Pinned objective loss/gradient equivalence failed")
        objective = json.loads(args.objective_check.read_text(encoding="utf-8"))
        if objective.get("status") != "PASS":
            raise RuntimeError("Objective receipt did not pass")
        shutil.copy2(args.objective_check, root / "objective_equivalence.json")

        reference_dir = root / "reference_eval"
        command = [sys.executable, str(HERE / "evaluate_npo_profile_cohort.py"), "--protocol", str(args.protocol), "--mode", "reference", "--output-dir", str(reference_dir)]
        if run_stage("full-checkpoint-reference-evaluation", command, root):
            raise RuntimeError("Full-checkpoint profile reference evaluation failed")
        reference_path = reference_dir / "full_reference_profile_eval.json"
        if not reference_path.is_file():
            raise RuntimeError("Reference evaluation receipt is missing")
        reference = json.loads(reference_path.read_text(encoding="utf-8"))
        if reference.get("status") != "PASS_PROFILE_COHORT_EVAL":
            raise RuntimeError("Full-checkpoint profile reference did not pass")

        smoke_dir = root / "smoke_alpha_5"
        command = [
            sys.executable, str(HERE / "npo_lora_train.py"), "--model", "3b", "--seed", "42",
            "--protocol", str(args.protocol), "--objective-check", str(args.objective_check),
            "--cohort", "dev", "--alpha-retain", "5", "--output-dir", str(smoke_dir), "--smoke",
        ]
        if run_stage("one-step-dev-cohort-smoke", command, root):
            raise RuntimeError("Real-data profile-cohort smoke failed")
        smoke = json.loads((smoke_dir / "training_receipt.json").read_text(encoding="utf-8"))
        if smoke.get("status") != "SMOKE_PASS" or smoke.get("global_steps") != 1 or smoke.get("smoke_max_abs_lora_parameter_update", 0) <= 0:
            raise RuntimeError("Real-data smoke receipt failed update checks")
        if smoke.get("runtime", {}).get("cpu_or_disk_offload") is not False:
            raise RuntimeError("Real-data smoke used CPU or disk offload")
        receipt["smoke_status"] = "PASS_BOUNDED_REAL_DATA_SMOKE"
        receipt["training_started"] = False
        write_json(receipt_path, receipt)

        selected_alpha = None
        for alpha in protocol["selection"]["candidate_retain_nll_weights_ascending"]:
            alpha_tag = f"{alpha:g}"
            train_dir = root / f"dev_alpha_{alpha_tag}"
            train_cmd = [
                sys.executable, str(HERE / "npo_lora_train.py"), "--model", "3b", "--seed", "42",
                "--protocol", str(args.protocol), "--objective-check", str(args.objective_check),
                "--cohort", "dev", "--alpha-retain", alpha_tag, "--output-dir", str(train_dir),
            ]
            receipt["training_started"] = True
            receipt["candidate_alpha_in_progress"] = alpha
            write_json(receipt_path, receipt)
            if run_stage(f"dev-training-alpha-{alpha_tag}", train_cmd, root):
                raise RuntimeError(f"Development NPO-LoRA training failed for alpha={alpha_tag}")
            train_receipt = json.loads((train_dir / "training_receipt.json").read_text(encoding="utf-8"))
            if train_receipt.get("status") != "TRAINING_COMPLETE" or train_receipt.get("global_steps") != protocol["training"]["expected_steps_per_cohort"]:
                raise RuntimeError(f"Development training receipt failed for alpha={alpha_tag}")
            eval_dir = root / "dev_eval"
            eval_cmd = [
                sys.executable, str(HERE / "evaluate_npo_profile_cohort.py"), "--protocol", str(args.protocol),
                "--mode", "dev", "--alpha-retain", alpha_tag, "--adapter-dir", str(train_dir / "adapter"), "--output-dir", str(eval_dir),
            ]
            if run_stage(f"dev-evaluation-alpha-{alpha_tag}", eval_cmd, root):
                raise RuntimeError(f"Development evaluation failed for alpha={alpha_tag}")
            comparison_path = eval_dir / f"dev_alpha_{alpha_tag}_comparison.json"
            comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
            attempt = {"alpha_retain": alpha, "training_dir": str(train_dir), "comparison": str(comparison_path), "gate_status": comparison["status"]}
            receipt["candidate_attempts"].append(attempt)
            write_json(receipt_path, receipt)
            if comparison.get("all_gates_pass") is True:
                selected_alpha = float(alpha)
                receipt["selected_alpha_retain"] = selected_alpha
                receipt["selection_rule"] = protocol["selection"]["rule"]
                write_json(receipt_path, receipt)
                break

        if selected_alpha is None:
            receipt["status"] = "NO_DEV_CANDIDATE_PASSES"
            receipt["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
            receipt["final_training_started"] = False
            write_json(receipt_path, receipt)
            return

        final_train_dir = root / f"final_alpha_{selected_alpha:g}"
        final_cmd = [
            sys.executable, str(HERE / "npo_lora_train.py"), "--model", "3b", "--seed", "42",
            "--protocol", str(args.protocol), "--objective-check", str(args.objective_check),
            "--cohort", "final", "--alpha-retain", f"{selected_alpha:g}", "--output-dir", str(final_train_dir),
        ]
        receipt["final_training_started"] = True
        write_json(receipt_path, receipt)
        if run_stage(f"final-training-alpha-{selected_alpha:g}", final_cmd, root):
            raise RuntimeError("Final cohort training failed")
        final_train_receipt = json.loads((final_train_dir / "training_receipt.json").read_text(encoding="utf-8"))
        if final_train_receipt.get("status") != "TRAINING_COMPLETE" or final_train_receipt.get("global_steps") != protocol["training"]["expected_steps_per_cohort"]:
            raise RuntimeError("Final training receipt failed")
        final_eval_dir = root / "final_eval"
        final_eval_cmd = [
            sys.executable, str(HERE / "evaluate_npo_profile_cohort.py"), "--protocol", str(args.protocol),
            "--mode", "final", "--alpha-retain", f"{selected_alpha:g}", "--adapter-dir", str(final_train_dir / "adapter"), "--output-dir", str(final_eval_dir),
        ]
        if run_stage("final-cohort-evaluation", final_eval_cmd, root):
            raise RuntimeError("Final cohort evaluation failed")
        final_comparison_path = final_eval_dir / f"final_alpha_{selected_alpha:g}_comparison.json"
        final_comparison = json.loads(final_comparison_path.read_text(encoding="utf-8"))
        receipt["final_comparison"] = str(final_comparison_path)
        receipt["status"] = final_comparison["status"]
        receipt["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        write_json(receipt_path, receipt)
    except Exception as error:
        receipt["status"] = "ERROR"
        receipt["error"] = f"{type(error).__name__}: {error}"
        receipt["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
        write_json(receipt_path, receipt)
        raise
    finally:
        if root.exists():
            artifact = archive(root, args.artifact_zip)
            receipt["artifact_archive"] = artifact
            write_json(receipt_path, receipt)
            print(json.dumps({"run_receipt": receipt, "artifact_archive": artifact}, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/npo_lora_recovery_protocol.json
{
  "protocol_id": "tofu_3b_npo_lora_profile_recovery_v1",
  "status": "PREREGISTERED_BEFORE_GPU_RUN",
  "study_type": "Exploratory post-hoc alpha-selection follow-up; one seed; not confirmatory.",
  "registered_locally_at": "2026-11-07",
  "relationship_to_prior_run": "Post-hoc follow-up prompted by the prior one-seed adapter run failing all four utility gates. This is diagnostic, not confirmatory, and does not change the prior result.",
  "research_question": "Can a stronger retain-NLL weight reduce utility damage from the previous 3B NPO-LoRA setup while preserving measurable forgetting on separate TOFU author-profile cohorts?",
  "model": {
    "model_family": "3b",
    "seed": 42,
    "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
    "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    "architecture_id": "meta-llama/Llama-3.2-3B-Instruct"
  },
  "objective": {
    "source": "locuslab/open-unlearning@2d98027677f3468d52b6baa3fc4f2b9f2dc08820",
    "beta": 0.1,
    "forget_weight_gamma": 1.0,
    "retain_loss_type": "token_mean_nll"
  },
  "data": {
    "repo_id": "locuslab/TOFU",
    "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
    "profile_block_size": 20,
    "expected_rows": {
      "forget10.json": 400,
      "forget10_perturbed.json": 400,
      "retain90.json": 3600,
      "holdout10.json": 400
    },
    "sha256": {
      "forget10.json": "0044c8c2e70a38be93f62ec6cb1c1cc2a1f55a8df2fb549a4da2da8dde9d92f6",
      "forget10_perturbed.json": "6fbcb946c57ea1d7b2124cea0e61bf3b5409d1bc10d02368f090109450ed73c7",
      "retain90.json": "debcf019c41db7b30b62ca6ad41fd8a4bcd29a5b321c21d2961974e308dfca55",
      "holdout10.json": "efec32c7a7f66bfefaa90b8a4cc583296b0bbf882495db61076dec2d6edf44dd"
    },
    "profile_assignment_ranges_inclusive": {
      "forget10_dev_train_and_eval": [[0, 9]],
      "forget10_final_train_and_eval": [[10, 19]],
      "retain90_optimizer_train": [[0, 4], [6, 21], [23, 59], [100, 179]],
      "retain90_dev_utility_eval": [[60, 79]],
      "retain90_final_utility_eval": [[80, 99]],
      "holdout10_dev_utility_eval": [[0, 9]],
      "holdout10_final_utility_eval": [[10, 19]],
      "retain90_excluded_conflicting_question_profiles": [5, 22]
    },
    "limitations": [
      "TOFU profile blocks are inferred from contiguous groups of 20 rows because the published files do not expose a structured author ID.",
      "The published full checkpoint was already fine-tuned on the TOFU training data; this protocol measures adapter effects relative to that checkpoint.",
      "This follow-up uses a subset of the benchmark and is not a standard full-split TOFU leaderboard run."
    ]
  },
  "selection": {
    "candidate_retain_nll_weights_ascending": [5.0, 10.0, 20.0],
    "rule": "Run candidates in ascending alpha_retain order and stop at the first candidate satisfying every development gate. If none passes, stop without running the final cohort.",
    "development_gates": {
      "forget10.truth_ratio_score": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > 0",
      "forget10.answer_probability": "upper bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be < 0",
      "retain90.answer_probability": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05",
      "retain90.rougeL_recall": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05",
      "holdout10.answer_probability": "lower bound of paired profile-block bootstrap 95% CI for candidate minus full checkpoint must be > -0.05"
    },
    "final_gates": "Apply the same five gates once to the disjoint final profile cohorts. No threshold changes after development results are observed.",
    "bootstrap": {
      "unit": "profile block, paired by source row index",
      "draws": 5000,
      "seed": 42,
      "interval": "percentile 95%"
    },
    "noninferiority_margin": 0.05
  },
  "training": {
    "method": "NPO forget objective plus weighted retain token-mean NLL, LoRA adapters only",
    "beta": 0.1,
    "forget_weight_gamma": 1.0,
    "learning_rate": 0.00001,
    "weight_decay": 0.0,
    "optimizer_betas": [0.9, 0.999],
    "optimizer_eps": 1e-8,
    "scheduler": "linear; one epoch warmup",
    "precision": "FP16 frozen base checkpoint; FP32 LoRA parameters and loss; SDPA",
    "gradient_clip_norm": 1.0,
    "epochs": 10,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "effective_paired_batch_size": 8,
    "retain_sampling": "uniform with replacement from optimizer-only retain profiles; seed 42",
    "max_length": 512,
    "truncation": false,
    "hardware": "Kaggle T4 x2 only; no CPU/disk offload; FP16 frozen base, FP32 LoRA and loss; SDPA",
    "expected_steps_per_cohort": 250,
    "expected_loss_trace_rows_per_cohort": 1000
  },
  "evaluation": {
    "metrics": ["answer_probability", "greedy-generation ROUGE-L recall", "forget Truth Ratio score"],
    "generations": "greedy, max_new_tokens=200",
    "dev_rows": {"forget10": 200, "retain90": 400, "holdout10": 200},
    "final_rows": {"forget10": 200, "retain90": 400, "holdout10": 200},
    "research_claim_limit": "One-seed exploratory diagnostic; final cohort is held out from alpha selection, but the result is not confirmatory."
  },
  "stopping_rules": [
    "Verify the pinned data hashes, row counts, profile assignments, and all disjointness checks before loading model weights.",
    "Verify the exact pinned NPO objective value and gradients before GPU training.",
    "Run candidates sequentially and stop at the first candidate that passes every development gate.",
    "If no candidate passes, do not run final-cohort training and report the negative development result.",
    "If a candidate passes development, run one final-cohort training/evaluation with its selected alpha and unchanged settings.",
    "Any runtime failure preserves logs and receipts; do not interpret a partial run as a research result."
  ]
}


In [ ]:
%%writefile /kaggle/working/requirements-npo-pilot.txt
# Kaggle runtime packages used by the bounded TOFU NPO-LoRA pilot.
# Keep the preinstalled CUDA-enabled torch build supplied by Kaggle.
transformers==5.22.1
accelerate==1.15.0
peft==0.21.2
scipy==1.16.3
rouge-score==0.1.2


In [ ]:
command = [sys.executable, '/kaggle/working/run_npo_profile_recovery.py']
subprocess.run(command, check=True)
artifact = Path('/kaggle/working/tofu_npo_profile_recovery_artifacts.zip')
if not artifact.is_file() or artifact.stat().st_size == 0: raise RuntimeError('Recovery result archive is missing')
import hashlib
print({'artifact_zip': str(artifact), 'bytes': artifact.stat().st_size, 'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest()})
